# Fabric Defect Detection — Faster R-CNN + ResNet-50 FPN

Classes: `0=background, 1=hole, 2=knot, 3=line, 4=stain` → `num_classes = 5`

**How to use this notebook on Kaggle:**
1. Add your dataset (`fabric_defect_detection_MERGED_CLEAN_4CLASS.zip`, or the extracted folder) as a data source.
2. Settings → Accelerator → GPU (P100 or T4), Internet → **ON** (needed once, to download COCO-pretrained weights).
3. Run cells top to bottom.

Cell 2 writes and runs the **dataset audit only** (no GPU, no training, dataset never modified).
Cell 3 writes the **training/eval pipeline** to disk.
Cell 4 is an optional 2-epoch **smoke test** on 40 images — proves everything runs end-to-end in a few minutes.
Cell 5 is the **real training run**. If a Kaggle session dies mid-run, just re-run Cell 5 — it resumes automatically from `last_model.pth`.


In [ ]:
# Environment check + install anything missing
import sys, subprocess
for pkg in ["pycocotools", "opencv-python-headless"]:
    try:
        __import__(pkg.split("-")[0].replace("opencv","cv2") if "opencv" in pkg else pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])
import torch, torchvision
print("PyTorch:", torch.__version__, "| Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          "|", round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2), "GB")


## 1. Dataset audit (run first — no GPU needed)

In [ ]:
%%writefile 01_inspect_dataset.py
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
01_inspect_dataset.py
=====================
STANDALONE dataset inspection / audit for the fabric defect detection dataset.

Run this FIRST, before training. It needs no GPU.

It does NOT modify the original dataset. It only reads, and writes a report to
/kaggle/working/faster_rcnn_results/dataset_audit.txt (+ .json)

Usage on Kaggle:
    !python 01_inspect_dataset.py
or paste into a notebook cell and call main().
"""

import os
import io
import sys
import json
import time
import zipfile
import hashlib
import platform
from collections import Counter, defaultdict

# ----------------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------------
SEARCH_ROOTS = ["/kaggle/input", "/kaggle/working", ".", "/content", "/mnt/data"]
EXTRACT_DIR = "/kaggle/working/dataset_extracted"
OUT_DIR = "/kaggle/working/faster_rcnn_results"
EXPECTED_CLASS_KEYWORDS = ["hole", "knot", "stain"]
IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp")

# hash-based duplicate detection can be slow on huge sets; cap it
DUPLICATE_CHECK_MAX_IMAGES = 20000


# ----------------------------------------------------------------------------
# small utils
# ----------------------------------------------------------------------------
def log(msg=""):
    print(msg, flush=True)


def hr(title=None):
    log("=" * 78)
    if title:
        log(title)
        log("=" * 78)


def human(n):
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if abs(n) < 1024.0:
            return f"{n:3.1f}{unit}"
        n /= 1024.0
    return f"{n:.1f}PB"


# ----------------------------------------------------------------------------
# 1. locate + extract the dataset
# ----------------------------------------------------------------------------
def find_zip_candidates(roots=SEARCH_ROOTS):
    found = []
    for root in roots:
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, filenames in os.walk(root):
            # do not descend into our own extraction dir
            if os.path.abspath(dirpath).startswith(os.path.abspath(EXTRACT_DIR)):
                dirnames[:] = []
                continue
            for fn in filenames:
                if fn.lower().endswith(".zip"):
                    found.append(os.path.join(dirpath, fn))
    return sorted(set(found))


def find_coco_jsons(root):
    """Walk a directory and return every JSON that looks like COCO detection format."""
    hits = []
    for dirpath, dirnames, filenames in os.walk(root):
        for fn in filenames:
            if not fn.lower().endswith(".json"):
                continue
            path = os.path.join(dirpath, fn)
            try:
                if os.path.getsize(path) > 500 * 1024 * 1024:
                    continue
                with open(path, "r", encoding="utf-8", errors="ignore") as f:
                    data = json.load(f)
            except Exception:
                continue
            if isinstance(data, dict) and "images" in data and "annotations" in data \
                    and "categories" in data:
                hits.append(path)
    return sorted(hits)


def guess_split(path):
    """Infer train/valid/test from the path string."""
    p = path.replace("\\", "/").lower()
    parts = p.split("/")
    # look at the deepest components first
    for seg in reversed(parts):
        if "train" in seg:
            return "train"
        if "valid" in seg or seg == "val" or "validation" in seg or seg.startswith("val_"):
            return "valid"
        if "test" in seg:
            return "test"
    return None


def locate_images_dir(json_path, coco):
    """
    Find the directory that actually contains the images referenced by this
    COCO json. Tries: json's own dir, ./images, parent, parent/images.
    """
    names = [im.get("file_name", "") for im in coco.get("images", [])][:50]
    names = [os.path.basename(n) for n in names if n]
    if not names:
        return os.path.dirname(json_path)

    jdir = os.path.dirname(json_path)
    candidates = [
        jdir,
        os.path.join(jdir, "images"),
        os.path.dirname(jdir),
        os.path.join(os.path.dirname(jdir), "images"),
    ]
    best, best_hits = jdir, -1
    for c in candidates:
        if not os.path.isdir(c):
            continue
        hits = sum(1 for n in names if os.path.isfile(os.path.join(c, n)))
        if hits > best_hits:
            best, best_hits = c, hits
    if best_hits > 0:
        return best

    # fall back: full recursive search for the first filename
    search_root = os.path.dirname(jdir) or jdir
    target = names[0]
    for dirpath, _, filenames in os.walk(search_root):
        if target in filenames:
            return dirpath
    return jdir


def discover_dataset(zip_path=None, verbose=True):
    """
    Returns dict:
      { 'root': str,
        'splits': { 'train': {'json': path, 'images_dir': path}, ... } }
    """
    root = None

    # (a) explicit / auto zip
    zips = [zip_path] if zip_path else find_zip_candidates()
    if verbose and zips:
        log("ZIP candidates found:")
        for z in zips:
            log(f"   {z}  ({human(os.path.getsize(z))})")

    # prefer a zip whose name mentions fabric/defect
    chosen_zip = None
    for z in zips:
        if z and os.path.isfile(z):
            base = os.path.basename(z).lower()
            if "fabric" in base or "defect" in base:
                chosen_zip = z
                break
    if chosen_zip is None and zips and zips[0] and os.path.isfile(zips[0]):
        chosen_zip = zips[0]

    if chosen_zip:
        os.makedirs(EXTRACT_DIR, exist_ok=True)
        marker = os.path.join(EXTRACT_DIR, "._extracted_" +
                              hashlib.md5(chosen_zip.encode()).hexdigest()[:10])
        if os.path.exists(marker):
            if verbose:
                log(f"\nAlready extracted -> {EXTRACT_DIR}")
        else:
            if verbose:
                log(f"\nExtracting {chosen_zip} -> {EXTRACT_DIR} ...")
            t0 = time.time()
            with zipfile.ZipFile(chosen_zip) as zf:
                zf.extractall(EXTRACT_DIR)
            open(marker, "w").write("ok")
            if verbose:
                log(f"Extracted in {time.time()-t0:.1f}s")
        root = EXTRACT_DIR

    # (b) no zip -> look for an already-extracted dataset dir
    if root is None:
        for r in SEARCH_ROOTS:
            if os.path.isdir(r) and find_coco_jsons(r):
                root = r
                break

    if root is None:
        raise FileNotFoundError(
            "Could not locate the dataset. Put the ZIP (or the extracted folder) "
            "somewhere under /kaggle/input or /kaggle/working, or pass zip_path=..."
        )

    jsons = find_coco_jsons(root)
    if not jsons:
        raise FileNotFoundError(f"No COCO-format JSON found under {root}")

    if verbose:
        log(f"\nCOCO JSON files detected under {root}:")
        for j in jsons:
            log(f"   {j}")

    splits = {}
    unknown = []
    for j in jsons:
        s = guess_split(j)
        if s is None:
            unknown.append(j)
            continue
        # if two jsons map to the same split, keep the one with more images
        with open(j, "r", encoding="utf-8", errors="ignore") as f:
            data = json.load(f)
        n_img = len(data.get("images", []))
        if s in splits and splits[s]["n_images"] >= n_img:
            continue
        splits[s] = {
            "json": j,
            "images_dir": locate_images_dir(j, data),
            "n_images": n_img,
        }

    if unknown and verbose:
        log("\n[warn] JSON files whose split could not be inferred from the path:")
        for u in unknown:
            log(f"   {u}")

    missing = [s for s in ("train", "valid", "test") if s not in splits]
    if "valid" in missing and "val" in splits:
        splits["valid"] = splits.pop("val")
        missing.remove("valid")
    if missing and verbose:
        log(f"\n[warn] splits not found: {missing}")

    return {"root": root, "splits": splits}


# ----------------------------------------------------------------------------
# 2. category mapping
# ----------------------------------------------------------------------------
def build_category_mapping(categories, verbose=True):
    """
    Map COCO category ids -> contiguous model labels:
        0 background, 1 hole, 2 knot, 3 stain
    Matching is done on normalised names; falls back to sorted-id order with a
    loud warning if the names are unrecognisable.
    """
    norm = []
    for c in categories:
        name = str(c.get("name", "")).strip().lower().replace("-", "_").replace(" ", "_")
        norm.append((c["id"], name, c.get("name")))

    coco_id_to_contig = {}
    contig_to_name = {}
    used = set()
    unmatched = []

    for cid, name, raw in norm:
        hit = None
        for i, kw in enumerate(EXPECTED_CLASS_KEYWORDS):
            if kw in name:
                hit = i + 1
                break
        if hit is not None and hit not in used:
            coco_id_to_contig[cid] = hit
            contig_to_name[hit] = EXPECTED_CLASS_KEYWORDS[hit - 1]
            used.add(hit)
        else:
            unmatched.append((cid, raw))

    if unmatched:
        if verbose:
            log("\n[warn] categories not matched by name: " + str(unmatched))
        free = [i for i in (1, 2, 3) if i not in used]
        for (cid, raw), slot in zip(sorted(unmatched), free):
            coco_id_to_contig[cid] = slot
            contig_to_name[slot] = str(raw)
            used.add(slot)

    # COCO supercategory rows sometimes carry id 0 with no real annotations;
    # anything still unmapped is recorded so the audit can flag it.
    contig_to_coco_id = {v: k for k, v in coco_id_to_contig.items()}
    return coco_id_to_contig, contig_to_coco_id, contig_to_name


# ----------------------------------------------------------------------------
# 3. audit one split
# ----------------------------------------------------------------------------
def audit_split(split_name, json_path, images_dir, do_hash=True, verbose=True):
    from PIL import Image
    Image.MAX_IMAGE_PIXELS = None

    with open(json_path, "r", encoding="utf-8", errors="ignore") as f:
        coco = json.load(f)

    images = coco.get("images", [])
    anns = coco.get("annotations", [])
    cats = coco.get("categories", [])

    coco_id_to_contig, contig_to_coco_id, contig_to_name = build_category_mapping(cats, verbose=False)

    rep = {
        "split": split_name,
        "json": json_path,
        "images_dir": images_dir,
        "n_images_declared": len(images),
        "n_annotations_declared": len(anns),
        "categories": [{"id": c["id"], "name": c.get("name"),
                        "contiguous_label": coco_id_to_contig.get(c["id"])} for c in cats],
    }

    img_by_id = {}
    duplicate_image_ids = []
    for im in images:
        if im["id"] in img_by_id:
            duplicate_image_ids.append(im["id"])
        img_by_id[im["id"]] = im

    # --- files on disk ---
    missing_files = []
    corrupted = []
    dim_mismatch = []
    sizes = Counter()
    widths, heights = [], []
    hashes = defaultdict(list)

    for im in images:
        fn = os.path.basename(im.get("file_name", ""))
        path = os.path.join(images_dir, fn)
        if not os.path.isfile(path):
            missing_files.append(fn)
            continue
        try:
            with Image.open(path) as pil:
                pil.verify()
            with Image.open(path) as pil:
                w, h = pil.size
        except Exception as e:
            corrupted.append((fn, repr(e)[:120]))
            continue
        widths.append(w); heights.append(h)
        sizes[(w, h)] += 1
        jw, jh = im.get("width"), im.get("height")
        if jw and jh and (jw != w or jh != h):
            dim_mismatch.append({"file": fn, "json": [jw, jh], "actual": [w, h]})
        if do_hash and len(images) <= DUPLICATE_CHECK_MAX_IMAGES:
            try:
                with open(path, "rb") as fh:
                    hashes[hashlib.md5(fh.read()).hexdigest()].append(fn)
            except Exception:
                pass

    duplicate_files = {h: fs for h, fs in hashes.items() if len(fs) > 1}

    # --- annotations ---
    invalid_boxes = []
    clipped_boxes = []
    bad_category = []
    crowd = 0
    per_class = Counter()
    per_image_count = Counter()
    areas = []
    aspect = []
    seen_ann = set()
    duplicate_anns = []
    valid_anns = 0

    for a in anns:
        aid = a.get("id")
        img_id = a.get("image_id")
        cid = a.get("category_id")
        bbox = a.get("bbox")

        key = (img_id, cid, tuple(round(float(v), 2) for v in bbox) if bbox else None)
        if key in seen_ann:
            duplicate_anns.append(aid)
        seen_ann.add(key)

        if cid not in coco_id_to_contig:
            bad_category.append({"ann_id": aid, "category_id": cid})
            continue
        if a.get("iscrowd", 0):
            crowd += 1
        if img_id not in img_by_id:
            invalid_boxes.append({"ann_id": aid, "reason": "image_id not in images"})
            continue
        if not bbox or len(bbox) != 4:
            invalid_boxes.append({"ann_id": aid, "reason": "bbox malformed"})
            continue

        x, y, w, h = [float(v) for v in bbox]
        im = img_by_id[img_id]
        W = float(im.get("width") or 0) or 1e9
        H = float(im.get("height") or 0) or 1e9

        if w <= 0 or h <= 0:
            invalid_boxes.append({"ann_id": aid, "reason": f"non-positive w/h ({w:.2f},{h:.2f})"})
            continue
        x1, y1, x2, y2 = x, y, x + w, y + h
        if x1 < -1 or y1 < -1 or x2 > W + 1 or y2 > H + 1:
            clipped_boxes.append({"ann_id": aid, "box": [x1, y1, x2, y2], "img": [W, H]})
        cx1, cy1 = max(0.0, x1), max(0.0, y1)
        cx2, cy2 = min(W, x2), min(H, y2)
        if cx2 - cx1 <= 1e-3 or cy2 - cy1 <= 1e-3:
            invalid_boxes.append({"ann_id": aid, "reason": "zero area after clipping"})
            continue

        valid_anns += 1
        lab = coco_id_to_contig[cid]
        per_class[contig_to_name.get(lab, str(lab))] += 1
        per_image_count[img_id] += 1
        bw, bh = cx2 - cx1, cy2 - cy1
        areas.append(bw * bh)
        aspect.append(bw / max(bh, 1e-6))

    empty_images = [im["id"] for im in images if per_image_count.get(im["id"], 0) == 0]

    def pct(vals, q):
        if not vals:
            return 0.0
        s = sorted(vals)
        i = min(len(s) - 1, max(0, int(round(q * (len(s) - 1)))))
        return float(s[i])

    rep.update({
        "n_images_on_disk": len(images) - len(missing_files) - len(corrupted),
        "missing_image_files": missing_files,
        "corrupted_images": corrupted,
        "duplicate_image_ids": duplicate_image_ids,
        "duplicate_image_files": duplicate_files,
        "json_vs_actual_dimension_mismatch": dim_mismatch[:50],
        "n_dimension_mismatch": len(dim_mismatch),
        "unique_resolutions": sorted(
            [{"wh": list(k), "count": v} for k, v in sizes.items()],
            key=lambda d: -d["count"])[:20],
        "width_min_med_max": [min(widths) if widths else 0, pct(widths, 0.5),
                              max(widths) if widths else 0],
        "height_min_med_max": [min(heights) if heights else 0, pct(heights, 0.5),
                               max(heights) if heights else 0],
        "n_valid_annotations": valid_anns,
        "n_invalid_boxes": len(invalid_boxes),
        "invalid_boxes_sample": invalid_boxes[:25],
        "n_boxes_needing_clip": len(clipped_boxes),
        "clipped_boxes_sample": clipped_boxes[:25],
        "n_bad_category": len(bad_category),
        "bad_category_sample": bad_category[:25],
        "n_iscrowd": crowd,
        "n_duplicate_annotations": len(duplicate_anns),
        "class_distribution": dict(per_class),
        "n_images_without_annotations": len(empty_images),
        "objects_per_image": {
            "mean": round(sum(per_image_count.values()) / max(1, len(images)), 3),
            "max": max(per_image_count.values()) if per_image_count else 0,
            "p50": pct(list(per_image_count.values()) + [0] * len(empty_images), 0.50),
            "p95": pct(list(per_image_count.values()) + [0] * len(empty_images), 0.95),
        },
        "box_area_px": {
            "p05": round(pct(areas, 0.05), 1), "p50": round(pct(areas, 0.50), 1),
            "p95": round(pct(areas, 0.95), 1),
            "min": round(min(areas), 1) if areas else 0,
            "max": round(max(areas), 1) if areas else 0,
        },
        "box_aspect_ratio_w_over_h": {
            "p05": round(pct(aspect, 0.05), 3), "p50": round(pct(aspect, 0.50), 3),
            "p95": round(pct(aspect, 0.95), 3),
        },
        "small_objects_lt_32x32": int(sum(1 for a in areas if a < 32 * 32)),
        "tiny_objects_lt_16x16": int(sum(1 for a in areas if a < 16 * 16)),
    })
    return rep


def print_report(rep):
    hr(f"SPLIT: {rep['split'].upper()}")
    log(f"  json            : {rep['json']}")
    log(f"  images_dir      : {rep['images_dir']}")
    log(f"  images (json)   : {rep['n_images_declared']}")
    log(f"  images on disk  : {rep['n_images_on_disk']}")
    log(f"  annotations     : {rep['n_annotations_declared']} declared / "
        f"{rep['n_valid_annotations']} valid")
    log("")
    log("  categories:")
    for c in rep["categories"]:
        log(f"     coco_id={c['id']:<4} name={str(c['name']):<12} -> label {c['contiguous_label']}")
    log("")
    log("  class distribution (valid annotations):")
    total = max(1, sum(rep["class_distribution"].values()))
    for k, v in sorted(rep["class_distribution"].items(), key=lambda x: -x[1]):
        log(f"     {k:<10} {v:>7}   ({100.0*v/total:5.2f}%)")
    log("")
    log(f"  images without annotations : {rep['n_images_without_annotations']}")
    log(f"  objects/image              : mean={rep['objects_per_image']['mean']} "
        f"p50={rep['objects_per_image']['p50']} p95={rep['objects_per_image']['p95']} "
        f"max={rep['objects_per_image']['max']}")
    log(f"  box area px (p05/p50/p95)  : {rep['box_area_px']['p05']} / "
        f"{rep['box_area_px']['p50']} / {rep['box_area_px']['p95']}")
    log(f"  boxes < 32x32 px           : {rep['small_objects_lt_32x32']}")
    log(f"  boxes < 16x16 px           : {rep['tiny_objects_lt_16x16']}")
    log(f"  aspect ratio w/h p05/50/95 : {rep['box_aspect_ratio_w_over_h']['p05']} / "
        f"{rep['box_aspect_ratio_w_over_h']['p50']} / {rep['box_aspect_ratio_w_over_h']['p95']}")
    log(f"  width  min/med/max         : {rep['width_min_med_max']}")
    log(f"  height min/med/max         : {rep['height_min_med_max']}")
    log("")
    log("  ISSUES")
    log(f"     missing image files     : {len(rep['missing_image_files'])}")
    log(f"     corrupted images        : {len(rep['corrupted_images'])}")
    log(f"     duplicate image ids     : {len(rep['duplicate_image_ids'])}")
    log(f"     duplicate image files   : {len(rep['duplicate_image_files'])}")
    log(f"     dimension mismatches    : {rep['n_dimension_mismatch']}")
    log(f"     invalid boxes (dropped) : {rep['n_invalid_boxes']}")
    log(f"     boxes needing clipping  : {rep['n_boxes_needing_clip']}")
    log(f"     bad category ids        : {rep['n_bad_category']}")
    log(f"     iscrowd annotations     : {rep['n_iscrowd']}")
    log(f"     duplicate annotations   : {rep['n_duplicate_annotations']}")
    if rep["missing_image_files"][:5]:
        log(f"     e.g. missing: {rep['missing_image_files'][:5]}")
    if rep["invalid_boxes_sample"][:3]:
        log(f"     e.g. invalid: {rep['invalid_boxes_sample'][:3]}")
    log("")


def check_cross_split_leakage(reports):
    """Same image file appearing in more than one split is a leak."""
    hr("CROSS-SPLIT LEAKAGE CHECK (identical file content across splits)")
    by_split = {}
    for r in reports:
        h = {}
        for hh, files in r.get("duplicate_image_files", {}).items():
            h[hh] = files
        by_split[r["split"]] = r
    # we need full hash sets -> recompute cheaply from disk
    import hashlib as _h
    split_hashes = {}
    for r in reports:
        d = r["images_dir"]
        hs = {}
        if os.path.isdir(d):
            for fn in os.listdir(d):
                if fn.lower().endswith(IMG_EXTS):
                    p = os.path.join(d, fn)
                    try:
                        with open(p, "rb") as f:
                            hs.setdefault(_h.md5(f.read()).hexdigest(), []).append(fn)
                    except Exception:
                        pass
        split_hashes[r["split"]] = hs

    names = list(split_hashes.keys())
    leaks = {}
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a, b = names[i], names[j]
            common = set(split_hashes[a]) & set(split_hashes[b])
            if common:
                leaks[f"{a}|{b}"] = [
                    (split_hashes[a][c][0], split_hashes[b][c][0]) for c in list(common)[:20]
                ]
                log(f"  [LEAK] {len(common)} identical images shared between {a} and {b}")
    if not leaks:
        log("  OK - no identical images shared across splits.")
    log("")
    return leaks


def main(zip_path=None):
    t0 = time.time()
    os.makedirs(OUT_DIR, exist_ok=True)

    hr("ENVIRONMENT")
    log(f"  python   : {platform.python_version()}")
    log(f"  platform : {platform.platform()}")
    try:
        from PIL import Image as _I
        log(f"  Pillow   : {_I.__version__}")
    except Exception:
        log("  Pillow   : NOT AVAILABLE (pip install pillow)")
    log("")

    hr("DATASET DISCOVERY")
    info = discover_dataset(zip_path=zip_path)
    log(f"\nResolved root: {info['root']}")
    for s, d in info["splits"].items():
        log(f"  {s:<6} json={d['json']}")
        log(f"  {'':<6} imgs={d['images_dir']}")
    log("")

    reports = []
    for split in ("train", "valid", "test"):
        if split not in info["splits"]:
            log(f"[warn] split '{split}' missing - skipped")
            continue
        d = info["splits"][split]
        rep = audit_split(split, d["json"], d["images_dir"])
        print_report(rep)
        reports.append(rep)

    leaks = check_cross_split_leakage(reports)

    hr("SUMMARY")
    log(f"{'split':<8}{'images':>9}{'anns':>9}{'hole':>8}{'knot':>8}{'stain':>8}{'empty':>8}")
    for r in reports:
        cd = r["class_distribution"]
        log(f"{r['split']:<8}{r['n_images_declared']:>9}{r['n_valid_annotations']:>9}"
            f"{cd.get('hole',0):>8}{cd.get('knot',0):>8}"
            f"{cd.get('stain',0):>8}{r['n_images_without_annotations']:>8}")
    log("")

    out_json = os.path.join(OUT_DIR, "dataset_audit.json")
    with open(out_json, "w") as f:
        json.dump({"root": info["root"],
                   "splits": {k: {kk: vv for kk, vv in v.items()} for k, v in info["splits"].items()},
                   "reports": reports,
                   "cross_split_leaks": leaks}, f, indent=2, default=str)
    log(f"Audit JSON written to {out_json}")
    log(f"Done in {time.time()-t0:.1f}s")
    return info, reports


if __name__ == "__main__":
    zp = sys.argv[1] if len(sys.argv) > 1 else None
    main(zp)


In [ ]:
# Runs the audit: finds the zip, extracts it, detects train/valid/test,
# checks classes, boxes, duplicates, missing files, class balance, etc.
# Read this output before training.
!python 01_inspect_dataset.py


## 2. Write the training/evaluation pipeline

In [ ]:
%%writefile fabric_frcnn_pipeline.py
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
================================================================================
 FABRIC DEFECT DETECTION  -  Faster R-CNN + ResNet-50 FPN  (PyTorch/Torchvision)
================================================================================
Classes: 0=background, 1=hole, 2=knot, 3=stain   ->  num_classes = 4

This file is SELF-CONTAINED. On Kaggle:

    !python fabric_frcnn_pipeline.py

or, in a notebook cell:

    from fabric_frcnn_pipeline import main
    main()

It will:
  1. print the environment (torch/torchvision/CUDA/GPU/memory)
  2. auto-discover the dataset (zip or folder), auto-detect train/valid/test and
     the COCO JSONs - nothing about the layout is hardcoded
  3. audit + clean annotations IN MEMORY (the dataset on disk is never modified)
  4. run a sanity test (GT visualisation, one forward/backward step, finite loss,
     output shape check) - full training only starts if it passes
  5. train Experiment A (800/1333) and optionally Experiment B (960/1600)
  6. evaluate on validation every epoch: per-class AP@50 / AP@50:95 /
     precision / recall / F1 + overall
  7. select the best checkpoint using validation mAP@50 *guarded against
     single-class collapse*, plus mAP@50:95 and F1
  8. sweep the confidence threshold on VALIDATION ONLY
  9. run exactly ONE final evaluation on the test set
 10. write metrics CSVs, plots, prediction images, error analysis, final report

No metric in the output is estimated or invented - everything printed is
computed from real model predictions by pycocotools + explicit TP/FP/FN matching.
================================================================================
"""

import os
import sys
import json
import math
import time
import shutil
import random
import zipfile
import hashlib
import platform
import argparse
import warnings
import traceback
from collections import Counter, defaultdict

warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np

# ==============================================================================
# CONFIG
# ==============================================================================
CONFIG = {
    # ---- data ----
    "zip_path": None,                 # None -> auto-search
    "search_roots": ["/kaggle/input", "/kaggle/working", ".", "/content", "/mnt/data"],
    "extract_dir": "/kaggle/working/dataset_extracted",
    "out_dir": "/kaggle/working/faster_rcnn_results",
    "class_keywords": ["hole", "knot", "stain"],   # -> labels 1..3

    # ---- model ----
    "num_classes": 4,                 # background + 3 defect classes (NEVER 3)
    "trainable_backbone_layers": 5,   # 5 = fine-tune the whole ResNet-50 (not frozen)
    "anchor_sizes": None,             # None -> torchvision default ((32,),(64,),(128,),(256,),(512,))
    "aspect_ratios": None,  # None -> torchvision default (0.5, 1.0, 2.0); hole/knot/stain
                            # are all roughly blob-shaped, so the extra-wide/tall anchors
                            # used for an elongated defect class are not needed here.
                            # Check the dataset audit's box aspect-ratio percentiles - if
                            # any class runs noticeably thin/elongated, widen this tuple.
    "box_detections_per_img": 100,
    "box_score_thresh_infer": 0.01,   # keep low; thresholding happens in metrics

    # ---- experiments (only these two, controlled) ----
    "experiments": [
        {"name": "expA", "min_size": 800, "max_size": 1333, "lr": 1e-4},
        {"name": "expB", "min_size": 960, "max_size": 1600, "lr": 1e-4},
    ],
    "run_experiment_B": True,

    # ---- optimisation ----
    "optimizer": "AdamW",
    "lr": 1e-4,
    "backbone_lr_mult": 0.1,          # backbone learns slower than the heads
    "weight_decay": 1e-4,
    "epochs": 70,                     # 60-80 with early stopping
    "warmup_epochs": 3,
    "min_lr_ratio": 0.02,             # cosine floor = lr * this
    "grad_clip_norm": 10.0,
    "amp": True,
    "batch_size": None,               # None -> auto from GPU memory
    "max_batch_size": 4,
    "grad_accum_steps": None,         # None -> auto so effective batch >= 4
    "early_stop_patience": 12,        # 10-15
    "seed": 42,

    # ---- augmentation (train only) ----
    "aug": {
        "hflip_p": 0.5,
        "vflip_p": 0.5,
        "affine_p": 0.5,
        "rotate_deg": 10.0,
        "scale_range": (0.90, 1.10),
        "translate_frac": 0.05,
        "brightness_contrast_p": 0.5,
        "brightness_delta": 0.18,     # multiplicative
        "contrast_delta": 0.18,
        "saturation_p": 0.30,
        "saturation_delta": 0.15,
        "gauss_noise_p": 0.20,
        "gauss_noise_sigma": 6.0,     # on 0-255 scale
        "min_box_side_px": 2.0,
        "min_visibility": 0.25,
    },

    # ---- evaluation ----
    "conf_thresholds": [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60],
    "iou_for_prf": 0.50,
    "eval_every": 1,

    # ---- checkpoint selection weights (validation only) ----
    "sel_w_map50": 0.55,
    "sel_w_weakest": 0.30,
    "sel_w_map5095": 0.15,
    "collapse_guard_ratio": 0.60,     # reject if weakest-class AP50 drops below
                                      # 60% of the best-so-far weakest AP50

    # ---- visualisation / robustness ----
    "n_vis_val": 20,
    "n_vis_test": 20,
    "n_err_examples": 20,
    "checkpoint_every": 5,
    "max_train_hours": 8.5,           # Kaggle session guard
    "num_workers": None,              # None -> auto
    "debug_subset": 0,                # >0 -> use only N train images (smoke test)
}

CLASS_NAMES = {1: "hole", 2: "knot", 3: "stain"}
CLASS_COLORS = {1: (255, 80, 80), 2: (80, 200, 255), 3: (170, 120, 255)}
IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp")


# ==============================================================================
# logging helpers
# ==============================================================================
_LOG_LINES = []


def log(msg=""):
    s = str(msg)
    print(s, flush=True)
    _LOG_LINES.append(s)


def hr(title=None, ch="="):
    log(ch * 78)
    if title:
        log(title)
        log(ch * 78)


def human(n):
    for u in ["B", "KB", "MB", "GB", "TB"]:
        if abs(n) < 1024.0:
            return f"{n:3.1f}{u}"
        n /= 1024.0
    return f"{n:.1f}PB"


def fmt_hms(sec):
    sec = int(sec)
    return f"{sec//3600:02d}:{(sec%3600)//60:02d}:{sec%60:02d}"


# ==============================================================================
# 0. ENVIRONMENT
# ==============================================================================
def setup_environment():
    import torch
    import torchvision

    hr("ENVIRONMENT")
    log(f"  python        : {platform.python_version()}")
    log(f"  platform      : {platform.platform()}")
    log(f"  PyTorch       : {torch.__version__}")
    log(f"  Torchvision   : {torchvision.__version__}")
    log(f"  CUDA available: {torch.cuda.is_available()}")
    log(f"  CUDA version  : {torch.version.cuda}")
    log(f"  cuDNN         : {torch.backends.cudnn.version() if torch.cuda.is_available() else 'n/a'}")

    gpu_mem_gb = 0.0
    if torch.cuda.is_available():
        n = torch.cuda.device_count()
        log(f"  GPU count     : {n}")
        for i in range(n):
            p = torch.cuda.get_device_properties(i)
            log(f"    [{i}] {p.name} | {p.total_memory/1024**3:.2f} GB | SM {p.major}.{p.minor}")
        gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
        torch.backends.cudnn.benchmark = True
    else:
        log("  [warn] No CUDA GPU detected. Training will be extremely slow on CPU.")
    log(f"  CPU count     : {os.cpu_count()}")

    try:
        import pycocotools
        log("  pycocotools   : available")
    except ImportError:
        log("  pycocotools   : MISSING -> attempting install")
        os.system(f"{sys.executable} -m pip install -q pycocotools")

    try:
        import cv2
        log(f"  OpenCV        : {cv2.__version__}")
    except ImportError:
        log("  OpenCV        : MISSING -> attempting install")
        os.system(f"{sys.executable} -m pip install -q opencv-python-headless")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    log(f"  device        : {device}")
    log("")
    return device, gpu_mem_gb


def seed_everything(seed):
    import torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def auto_batch_size(gpu_mem_gb, min_size):
    """Conservative: resolution is never sacrificed for batch size."""
    if gpu_mem_gb <= 0:
        return 1
    scale = (800.0 / max(min_size, 1)) ** 2
    if gpu_mem_gb >= 30:
        bs = 4
    elif gpu_mem_gb >= 15:
        bs = 4 if scale >= 1.0 else 2
    elif gpu_mem_gb >= 10:
        bs = 2
    else:
        bs = 1
    return max(1, min(CONFIG["max_batch_size"], bs))


def auto_num_workers():
    if CONFIG["num_workers"] is not None:
        return CONFIG["num_workers"]
    c = os.cpu_count() or 2
    return max(0, min(4, c - 1)) if c > 1 else 0


# ==============================================================================
# 1. DATASET DISCOVERY  (nothing assumed)
# ==============================================================================
def find_zip_candidates(roots):
    found = []
    for root in roots:
        if not os.path.isdir(root):
            continue
        for dp, dn, fn in os.walk(root):
            if os.path.abspath(dp).startswith(os.path.abspath(CONFIG["extract_dir"])):
                dn[:] = []
                continue
            for f in fn:
                if f.lower().endswith(".zip"):
                    found.append(os.path.join(dp, f))
    return sorted(set(found))


def find_coco_jsons(root):
    hits = []
    for dp, _, fns in os.walk(root):
        for f in fns:
            if not f.lower().endswith(".json"):
                continue
            p = os.path.join(dp, f)
            try:
                if os.path.getsize(p) > 500 * 1024 * 1024:
                    continue
                with open(p, "r", encoding="utf-8", errors="ignore") as fh:
                    d = json.load(fh)
            except Exception:
                continue
            if isinstance(d, dict) and {"images", "annotations", "categories"} <= set(d.keys()):
                hits.append(p)
    return sorted(hits)


def guess_split(path):
    p = path.replace("\\", "/").lower()
    for seg in reversed(p.split("/")):
        if "train" in seg:
            return "train"
        if "valid" in seg or "validation" in seg or seg == "val" or seg.startswith("val_") or seg.startswith("val."):
            return "valid"
        if "test" in seg:
            return "test"
    return None


def locate_images_dir(json_path, coco):
    names = [os.path.basename(im.get("file_name", "")) for im in coco.get("images", [])][:50]
    names = [n for n in names if n]
    jdir = os.path.dirname(json_path)
    if not names:
        return jdir
    cands = [jdir, os.path.join(jdir, "images"), os.path.dirname(jdir),
             os.path.join(os.path.dirname(jdir), "images")]
    best, best_hits = jdir, -1
    for c in cands:
        if not os.path.isdir(c):
            continue
        h = sum(1 for n in names if os.path.isfile(os.path.join(c, n)))
        if h > best_hits:
            best, best_hits = c, h
    if best_hits > 0:
        return best
    root = os.path.dirname(jdir) or jdir
    for dp, _, fns in os.walk(root):
        if names[0] in fns:
            return dp
    return jdir


def discover_dataset(zip_path=None):
    hr("DATASET DISCOVERY")
    root = None
    zips = [zip_path] if zip_path else find_zip_candidates(CONFIG["search_roots"])
    zips = [z for z in zips if z and os.path.isfile(z)]
    if zips:
        log("  ZIP candidates:")
        for z in zips:
            log(f"    {z}  ({human(os.path.getsize(z))})")
    chosen = None
    for z in zips:
        b = os.path.basename(z).lower()
        if "fabric" in b or "defect" in b:
            chosen = z
            break
    if chosen is None and zips:
        chosen = zips[0]

    if chosen:
        os.makedirs(CONFIG["extract_dir"], exist_ok=True)
        marker = os.path.join(CONFIG["extract_dir"],
                              "._ok_" + hashlib.md5(chosen.encode()).hexdigest()[:10])
        if os.path.exists(marker):
            log(f"  already extracted -> {CONFIG['extract_dir']}")
        else:
            log(f"  extracting {chosen} ...")
            t0 = time.time()
            with zipfile.ZipFile(chosen) as zf:
                zf.extractall(CONFIG["extract_dir"])
            open(marker, "w").write("ok")
            log(f"  extracted in {time.time()-t0:.1f}s")
        root = CONFIG["extract_dir"]

    if root is None:
        for r in CONFIG["search_roots"]:
            if os.path.isdir(r) and find_coco_jsons(r):
                root = r
                break
    if root is None:
        raise FileNotFoundError("Dataset not found. Place the ZIP or extracted folder under "
                                "/kaggle/input (or set CONFIG['zip_path']).")

    jsons = find_coco_jsons(root)
    if not jsons:
        raise FileNotFoundError(f"No COCO JSON under {root}")
    log(f"\n  root: {root}")
    log("  COCO JSONs found:")
    for j in jsons:
        log(f"    [{guess_split(j) or '??'}] {j}")

    splits = {}
    for j in jsons:
        s = guess_split(j)
        if s is None:
            continue
        with open(j, "r", encoding="utf-8", errors="ignore") as f:
            d = json.load(f)
        n = len(d.get("images", []))
        if s in splits and splits[s]["n_images"] >= n:
            continue
        splits[s] = {"json": j, "images_dir": locate_images_dir(j, d), "n_images": n}

    for s in ("train", "valid", "test"):
        if s in splits:
            log(f"\n  {s:<6}: {splits[s]['n_images']} images")
            log(f"          json = {splits[s]['json']}")
            log(f"          imgs = {splits[s]['images_dir']}")
        else:
            log(f"\n  [warn] split '{s}' NOT FOUND")
    log("")
    if "train" not in splits or "valid" not in splits:
        raise RuntimeError("train and valid splits are required.")
    return root, splits


def build_category_mapping(categories):
    coco2contig, contig2name = {}, {}
    used, unmatched = set(), []
    for c in categories:
        raw = c.get("name", "")
        name = str(raw).strip().lower().replace("-", "_").replace(" ", "_")
        hit = None
        for i, kw in enumerate(CONFIG["class_keywords"]):
            if kw in name:
                hit = i + 1
                break
        if hit is not None and hit not in used:
            coco2contig[c["id"]] = hit
            contig2name[hit] = CONFIG["class_keywords"][hit - 1]
            used.add(hit)
        else:
            unmatched.append((c["id"], raw))
    if unmatched:
        log(f"  [warn] categories not matched by name: {unmatched}")
        free = [i for i in (1, 2, 3) if i not in used]
        for (cid, raw), slot in zip(sorted(unmatched), free):
            coco2contig[cid] = slot
            contig2name[slot] = str(raw)
            used.add(slot)
            log(f"  [warn] forced mapping coco_id={cid} ('{raw}') -> label {slot}")
    contig2coco = {v: k for k, v in coco2contig.items()}
    return coco2contig, contig2coco, contig2name


# ==============================================================================
# 2. CLEANING + DATASET
# ==============================================================================
def load_and_clean_split(split, json_path, images_dir, verbose=True):
    """
    Returns records + a clean COCO-style GT dict (category_id = contiguous label).
    The dataset ON DISK IS NEVER MODIFIED - all cleaning is in memory.
    """
    from PIL import Image
    Image.MAX_IMAGE_PIXELS = None

    with open(json_path, "r", encoding="utf-8", errors="ignore") as f:
        coco = json.load(f)

    coco2contig, contig2coco, contig2name = build_category_mapping(coco["categories"])

    img_by_id, dup_ids = {}, 0
    for im in coco["images"]:
        if im["id"] in img_by_id:
            dup_ids += 1
        img_by_id[im["id"]] = im

    anns_by_img = defaultdict(list)
    stats = Counter()
    seen = set()
    for a in coco["annotations"]:
        cid = a.get("category_id")
        if cid not in coco2contig:
            stats["bad_category"] += 1
            continue
        if a.get("image_id") not in img_by_id:
            stats["orphan_annotation"] += 1
            continue
        bbox = a.get("bbox")
        if not bbox or len(bbox) != 4:
            stats["malformed_bbox"] += 1
            continue
        key = (a["image_id"], cid, tuple(round(float(v), 2) for v in bbox))
        if key in seen:
            stats["duplicate_annotation"] += 1
            continue
        seen.add(key)
        anns_by_img[a["image_id"]].append(a)

    records, gt_images, gt_anns = [], [], []
    ann_uid = 1
    missing, corrupt = [], []

    for img_id, im in img_by_id.items():
        fn = os.path.basename(im.get("file_name", ""))
        path = os.path.join(images_dir, fn)
        if not os.path.isfile(path):
            missing.append(fn)
            stats["missing_image"] += 1
            continue
        try:
            with Image.open(path) as pil:
                W, H = pil.size
                pil.draft(None, None)
        except Exception as e:
            corrupt.append((fn, repr(e)[:80]))
            stats["corrupted_image"] += 1
            continue

        jw, jh = im.get("width"), im.get("height")
        if jw and jh and (jw != W or jh != H):
            stats["dimension_mismatch"] += 1  # actual pixel size always wins

        boxes, labels = [], []
        for a in anns_by_img.get(img_id, []):
            x, y, w, h = [float(v) for v in a["bbox"]]
            if not all(map(math.isfinite, (x, y, w, h))):
                stats["invalid_nonfinite"] += 1
                continue
            if w <= 0 or h <= 0:
                stats["invalid_nonpositive_wh"] += 1
                continue
            x1, y1, x2, y2 = x, y, x + w, y + h
            if x1 > x2 or y1 > y2:
                stats["invalid_impossible_coords"] += 1
                continue
            needs_clip = (x1 < 0 or y1 < 0 or x2 > W or y2 > H)
            cx1, cy1 = max(0.0, x1), max(0.0, y1)
            cx2, cy2 = min(float(W), x2), min(float(H), y2)
            if needs_clip:
                stats["clipped"] += 1          # clipped, NOT deleted
            if (cx2 - cx1) < 1.0 or (cy2 - cy1) < 1.0:
                stats["invalid_zero_area"] += 1
                continue
            boxes.append([cx1, cy1, cx2, cy2])
            labels.append(coco2contig[a["category_id"]])

        records.append({
            "image_id": int(img_id),
            "file_name": fn,
            "path": path,
            "width": int(W),
            "height": int(H),
            "boxes": np.array(boxes, dtype=np.float32).reshape(-1, 4),
            "labels": np.array(labels, dtype=np.int64),
        })
        if not boxes:
            stats["image_without_annotations"] += 1   # kept as a negative sample

        gt_images.append({"id": int(img_id), "file_name": fn, "width": int(W), "height": int(H)})
        for b, l in zip(boxes, labels):
            gt_anns.append({
                "id": ann_uid, "image_id": int(img_id), "category_id": int(l),
                "bbox": [b[0], b[1], b[2] - b[0], b[3] - b[1]],
                "area": float((b[2] - b[0]) * (b[3] - b[1])), "iscrowd": 0,
            })
            ann_uid += 1

    records.sort(key=lambda r: r["image_id"])
    gt_dict = {
        "info": {"description": f"fabric defects - {split} (cleaned in memory)"},
        "licenses": [],
        "images": gt_images,
        "annotations": gt_anns,
        "categories": [{"id": k, "name": contig2name.get(k, CLASS_NAMES[k])} for k in (1, 2, 3)],
    }

    dist = Counter()
    for r in records:
        for l in r["labels"]:
            dist[int(l)] += 1

    if verbose:
        hr(f"SPLIT '{split}'  -  cleaning summary", "-")
        log(f"  images kept        : {len(records)} / {len(img_by_id)} declared")
        log(f"  annotations kept   : {len(gt_anns)} / {len(coco['annotations'])} declared")
        log(f"  duplicate image ids: {dup_ids}")
        log("  category mapping:")
        for c in coco["categories"]:
            log(f"    coco_id={c['id']:<4} '{c.get('name')}' -> label {coco2contig.get(c['id'])}")
        log("  class distribution:")
        tot = max(1, sum(dist.values()))
        for k in (1, 2, 3):
            n = dist.get(k, 0)
            log(f"    {contig2name.get(k, CLASS_NAMES[k]):<8} {n:>7}  ({100.0*n/tot:5.2f}%)")
        log("  issues:")
        for k in sorted(stats):
            log(f"    {k:<28} {stats[k]}")
        if missing[:5]:
            log(f"    e.g. missing files: {missing[:5]}")
        if corrupt[:3]:
            log(f"    e.g. corrupted    : {corrupt[:3]}")
        npi = [len(r["labels"]) for r in records]
        if npi:
            log(f"  objects/image      : mean={np.mean(npi):.2f} max={max(npi)} "
                f"p95={np.percentile(npi,95):.1f}")
        allb = np.concatenate([r["boxes"] for r in records if len(r["boxes"])], 0) \
            if any(len(r["boxes"]) for r in records) else np.zeros((0, 4), np.float32)
        if len(allb):
            bw = allb[:, 2] - allb[:, 0]
            bh = allb[:, 3] - allb[:, 1]
            ar = bw / np.maximum(bh, 1e-6)
            log(f"  box w  p05/50/95   : {np.percentile(bw,5):.1f}/{np.percentile(bw,50):.1f}/{np.percentile(bw,95):.1f}")
            log(f"  box h  p05/50/95   : {np.percentile(bh,5):.1f}/{np.percentile(bh,50):.1f}/{np.percentile(bh,95):.1f}")
            log(f"  aspect p05/50/95   : {np.percentile(ar,5):.2f}/{np.percentile(ar,50):.2f}/{np.percentile(ar,95):.2f}")
            log(f"  boxes < 32x32 px   : {int(((bw*bh) < 1024).sum())}")
        log("")

    return {
        "split": split, "records": records, "gt_dict": gt_dict,
        "coco2contig": coco2contig, "contig2coco": contig2coco, "contig2name": contig2name,
        "class_distribution": {int(k): int(v) for k, v in dist.items()},
        "clean_stats": dict(stats),
    }


# ------------------------------------------------------------------ augmentation
class FabricAugmenter:
    """
    cv2-based augmentation. No external dependency beyond OpenCV (always on Kaggle).
    Deliberately moderate: fabric defects must keep their appearance.
    Never applied to valid/test.
    """

    def __init__(self, cfg):
        self.c = cfg

    @staticmethod
    def _clean(boxes, labels, W, H, min_side, min_vis, orig_area=None):
        if len(boxes) == 0:
            return boxes, labels
        b = boxes.copy()
        area_before = orig_area if orig_area is not None else \
            np.maximum(b[:, 2] - b[:, 0], 0) * np.maximum(b[:, 3] - b[:, 1], 0)
        b[:, 0] = np.clip(b[:, 0], 0, W - 1)
        b[:, 1] = np.clip(b[:, 1], 0, H - 1)
        b[:, 2] = np.clip(b[:, 2], 0, W - 1)
        b[:, 3] = np.clip(b[:, 3], 0, H - 1)
        w = b[:, 2] - b[:, 0]
        h = b[:, 3] - b[:, 1]
        area_after = np.maximum(w, 0) * np.maximum(h, 0)
        vis = area_after / np.maximum(area_before, 1e-6)
        keep = (w >= min_side) & (h >= min_side) & (vis >= min_vis)
        return b[keep], labels[keep]

    def __call__(self, img, boxes, labels):
        import cv2
        c = self.c
        H, W = img.shape[:2]
        boxes = boxes.copy().astype(np.float32)
        labels = labels.copy()
        n_before = len(boxes)

        if random.random() < c["hflip_p"]:
            img = img[:, ::-1].copy()
            if len(boxes):
                x1 = boxes[:, 0].copy()
                boxes[:, 0] = W - boxes[:, 2]
                boxes[:, 2] = W - x1

        if random.random() < c["vflip_p"]:
            img = img[::-1].copy()
            if len(boxes):
                y1 = boxes[:, 1].copy()
                boxes[:, 1] = H - boxes[:, 3]
                boxes[:, 3] = H - y1

        if random.random() < c["affine_p"]:
            ang = random.uniform(-c["rotate_deg"], c["rotate_deg"])
            sc = random.uniform(*c["scale_range"])
            tx = random.uniform(-c["translate_frac"], c["translate_frac"]) * W
            ty = random.uniform(-c["translate_frac"], c["translate_frac"]) * H
            M = cv2.getRotationMatrix2D((W / 2.0, H / 2.0), ang, sc)
            M[0, 2] += tx
            M[1, 2] += ty
            img_t = cv2.warpAffine(img, M, (W, H), flags=cv2.INTER_LINEAR,
                                   borderMode=cv2.BORDER_REFLECT_101)
            if len(boxes):
                pts = np.stack([
                    boxes[:, [0, 1]], boxes[:, [2, 1]], boxes[:, [2, 3]], boxes[:, [0, 3]]
                ], axis=1)                                   # (N,4,2)
                ones = np.ones((*pts.shape[:2], 1), np.float32)
                pts_h = np.concatenate([pts, ones], -1)      # (N,4,3)
                tp = pts_h @ M.T                             # (N,4,2)
                nb = np.stack([tp[:, :, 0].min(1), tp[:, :, 1].min(1),
                               tp[:, :, 0].max(1), tp[:, :, 1].max(1)], 1).astype(np.float32)
                orig_area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1]) * (sc ** 2)
                nb, nl = self._clean(nb, labels, W, H, c["min_box_side_px"],
                                     c["min_visibility"], orig_area)
                # do not turn a positive image into an empty one through geometry
                if n_before > 0 and len(nb) == 0:
                    pass                                     # discard this affine
                else:
                    img, boxes, labels = img_t, nb, nl
            else:
                img = img_t

        img = img.astype(np.float32)
        if random.random() < c["brightness_contrast_p"]:
            b = 1.0 + random.uniform(-c["brightness_delta"], c["brightness_delta"])
            k = 1.0 + random.uniform(-c["contrast_delta"], c["contrast_delta"])
            mean = img.mean()
            img = (img - mean) * k + mean * b

        if random.random() < c["saturation_p"]:
            g = img.mean(axis=2, keepdims=True)
            s = 1.0 + random.uniform(-c["saturation_delta"], c["saturation_delta"])
            img = g + (img - g) * s

        if random.random() < c["gauss_noise_p"]:
            img = img + np.random.normal(0, c["gauss_noise_sigma"], img.shape).astype(np.float32)

        img = np.clip(img, 0, 255).astype(np.uint8)
        boxes, labels = self._clean(boxes, labels, img.shape[1], img.shape[0],
                                    c["min_box_side_px"], c["min_visibility"])
        return img, boxes, labels


class FabricDetectionDataset:
    """torch Dataset. Returns (image_tensor CHW float 0-1, target dict)."""

    def __init__(self, records, train=False, aug_cfg=None):
        self.records = records
        self.train = train
        self.aug = FabricAugmenter(aug_cfg) if (train and aug_cfg) else None

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        import torch
        import cv2
        r = self.records[idx]
        img = cv2.imread(r["path"], cv2.IMREAD_COLOR)
        if img is None:
            from PIL import Image
            img = np.array(Image.open(r["path"]).convert("RGB"))
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        boxes = r["boxes"].copy()
        labels = r["labels"].copy()

        if self.aug is not None:
            img, boxes, labels = self.aug(img, boxes, labels)

        t = torch.from_numpy(np.ascontiguousarray(img)).permute(2, 0, 1).float() / 255.0
        if len(boxes) == 0:
            boxes_t = torch.zeros((0, 4), dtype=torch.float32)
            labels_t = torch.zeros((0,), dtype=torch.int64)
            area_t = torch.zeros((0,), dtype=torch.float32)
        else:
            boxes_t = torch.as_tensor(boxes, dtype=torch.float32)
            labels_t = torch.as_tensor(labels, dtype=torch.int64)
            area_t = (boxes_t[:, 2] - boxes_t[:, 0]) * (boxes_t[:, 3] - boxes_t[:, 1])
        target = {
            "boxes": boxes_t,
            "labels": labels_t,
            "image_id": torch.tensor([r["image_id"]], dtype=torch.int64),
            "area": area_t,
            "iscrowd": torch.zeros((len(labels_t),), dtype=torch.int64),
        }
        return t, target


def collate_fn(batch):
    return tuple(zip(*batch))


def make_loader(records, train, batch_size, aug_cfg, device):
    import torch
    from torch.utils.data import DataLoader
    ds = FabricDetectionDataset(records, train=train, aug_cfg=aug_cfg if train else None)
    nw = auto_num_workers()
    kw = dict(batch_size=batch_size if train else 1,
              shuffle=train, num_workers=nw, collate_fn=collate_fn,
              pin_memory=(device.type == "cuda"), drop_last=False)
    if nw > 0:
        kw["persistent_workers"] = True
        kw["prefetch_factor"] = 2
    return DataLoader(ds, **kw)


# ==============================================================================
# 3. MODEL
# ==============================================================================
def build_model(num_classes, min_size, max_size, device):
    import torch
    import torchvision
    from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
    from torchvision.models.detection.rpn import AnchorGenerator

    kwargs = dict(min_size=min_size, max_size=max_size,
                  box_detections_per_img=CONFIG["box_detections_per_img"],
                  box_score_thresh=CONFIG["box_score_thresh_infer"],
                  trainable_backbone_layers=CONFIG["trainable_backbone_layers"])

    model = None
    try:
        from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights
        model = torchvision.models.detection.fasterrcnn_resnet50_fpn(
            weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1, **kwargs)
        log("  backbone weights : COCO_V1 pretrained (torchvision weights API)")
    except Exception as e1:
        try:
            model = torchvision.models.detection.fasterrcnn_resnet50_fpn(
                pretrained=True, **kwargs)
            log("  backbone weights : COCO pretrained (legacy pretrained=True)")
        except Exception as e2:
            model = torchvision.models.detection.fasterrcnn_resnet50_fpn(
                weights=None, weights_backbone=None, **kwargs)
            log(f"  [warn] could not download pretrained weights ({e1} / {e2}) "
                f"-> RANDOM INIT. Enable internet in Kaggle for much better results.")

    # anchor aspect ratios can be widened here for a thin/elongated defect class,
    # if the dataset ever has one; not used for hole/knot/stain
    if CONFIG["aspect_ratios"]:
        sizes = CONFIG["anchor_sizes"] or ((32,), (64,), (128,), (256,), (512,))
        ars = tuple([tuple(CONFIG["aspect_ratios"])] * len(sizes))
        model.rpn.anchor_generator = AnchorGenerator(sizes=sizes, aspect_ratios=ars)
        n_anchors = len(sizes[0]) * len(ars[0])
        in_ch = model.backbone.out_channels
        from torchvision.models.detection.rpn import RPNHead
        model.rpn.head = RPNHead(in_ch, n_anchors)
        log(f"  anchors          : sizes={sizes} aspect_ratios={CONFIG['aspect_ratios']} "
            f"({n_anchors} per location; RPN head re-initialised)")

    in_feat = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_feat, num_classes)

    assert model.roi_heads.box_predictor.cls_score.out_features == num_classes, \
        f"predictor head has {model.roi_heads.box_predictor.cls_score.out_features} " \
        f"outputs but num_classes={num_classes} was requested"
    assert num_classes == CONFIG["num_classes"], \
        f"build_model called with num_classes={num_classes} but CONFIG['num_classes']=" \
        f"{CONFIG['num_classes']} (background + {CONFIG['num_classes']-1} defect classes) - " \
        f"these must match"

    model.transform.min_size = (min_size,)
    model.transform.max_size = max_size

    model.to(device)
    n_par = sum(p.numel() for p in model.parameters())
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    log(f"  num_classes      : {num_classes}  (background + 3 defect classes)")
    log(f"  min_size/max_size: {min_size} / {max_size}")
    log(f"  parameters       : {n_par/1e6:.1f}M total, {n_tr/1e6:.1f}M trainable")
    return model


def build_optimizer(model, lr, wd):
    import torch
    backbone, heads = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (backbone if n.startswith("backbone") else heads).append(p)
    groups = [
        {"params": backbone, "lr": lr * CONFIG["backbone_lr_mult"], "name": "backbone"},
        {"params": heads, "lr": lr, "name": "heads"},
    ]
    opt = torch.optim.AdamW(groups, lr=lr, weight_decay=wd)
    log(f"  optimizer        : AdamW  lr(heads)={lr:g}  lr(backbone)={lr*CONFIG['backbone_lr_mult']:g}"
        f"  weight_decay={wd:g}")
    return opt


def build_scheduler(optimizer, steps_per_epoch, epochs):
    import torch
    warm = max(1, int(CONFIG["warmup_epochs"] * steps_per_epoch))
    total = max(warm + 1, epochs * steps_per_epoch)
    floor = CONFIG["min_lr_ratio"]

    def fn(step):
        if step < warm:
            return (step + 1) / warm
        prog = (step - warm) / max(1, total - warm)
        prog = min(1.0, prog)
        return floor + (1 - floor) * 0.5 * (1 + math.cos(math.pi * prog))

    log(f"  scheduler        : linear warm-up {CONFIG['warmup_epochs']} epoch(s) "
        f"-> cosine annealing to {floor:.0%} of base LR")
    return torch.optim.lr_scheduler.LambdaLR(optimizer, fn)


# ==============================================================================
# 4. METRICS
# ==============================================================================
def box_iou_np(a, b):
    """a (N,4), b (M,4) xyxy -> (N,M)"""
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), np.float32)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    lt = np.maximum(a[:, None, :2], b[None, :, :2])
    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = np.clip(rb - lt, 0, None)
    inter = wh[..., 0] * wh[..., 1]
    return (inter / np.maximum(area_a[:, None] + area_b[None, :] - inter, 1e-9)).astype(np.float32)


def run_inference(model, records, device, amp=True, desc="eval"):
    """
    Runs the model once over the records and caches raw predictions
    (so every confidence threshold can be evaluated without re-running).
    """
    import torch
    model.eval()
    preds = {}
    loader = make_loader(records, train=False, batch_size=1, aug_cfg=None, device=device)
    autocast = torch.cuda.amp.autocast if device.type == "cuda" else None
    t0 = time.time()
    n = 0
    with torch.no_grad():
        for imgs, targets in loader:
            imgs = [i.to(device, non_blocking=True) for i in imgs]
            if amp and autocast is not None:
                with autocast():
                    out = model(imgs)
            else:
                out = model(imgs)
            for t, o in zip(targets, out):
                iid = int(t["image_id"].item())
                preds[iid] = {
                    "boxes": o["boxes"].detach().float().cpu().numpy(),
                    "scores": o["scores"].detach().float().cpu().numpy(),
                    "labels": o["labels"].detach().cpu().numpy().astype(np.int64),
                }
            n += len(imgs)
    dt = time.time() - t0
    log(f"    [{desc}] inference on {n} images in {dt:.1f}s ({n/max(dt,1e-6):.2f} img/s)")
    return preds


def coco_eval_metrics(gt_dict, preds, score_thresh=0.0):
    """
    Real COCO AP via pycocotools. Returns overall + per-class AP@50 / AP@50:95
    and the PR curve arrays at IoU=0.50.
    """
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval

    coco_gt = COCO()
    coco_gt.dataset = gt_dict
    coco_gt.createIndex()

    dets = []
    for iid, p in preds.items():
        keep = p["scores"] >= score_thresh
        for b, s, l in zip(p["boxes"][keep], p["scores"][keep], p["labels"][keep]):
            dets.append({
                "image_id": int(iid), "category_id": int(l),
                "bbox": [float(b[0]), float(b[1]), float(b[2] - b[0]), float(b[3] - b[1])],
                "score": float(s),
            })

    out = {"mAP50": 0.0, "mAP5095": 0.0, "per_class": {}, "pr_curves": {}, "n_dets": len(dets)}
    if not dets:
        for k in (1, 2, 3):
            out["per_class"][k] = {"AP50": 0.0, "AP5095": 0.0}
        return out

    coco_dt = coco_gt.loadRes(dets)
    E = COCOeval(coco_gt, coco_dt, iouType="bbox")
    E.evaluate(); E.accumulate(); E.summarize()
    out["mAP5095"] = float(E.stats[0]) if not math.isnan(E.stats[0]) else 0.0
    out["mAP50"] = float(E.stats[1]) if not math.isnan(E.stats[1]) else 0.0
    out["AP_small"] = float(E.stats[3]) if not math.isnan(E.stats[3]) else float("nan")
    out["AP_medium"] = float(E.stats[4]) if not math.isnan(E.stats[4]) else float("nan")
    out["AP_large"] = float(E.stats[5]) if not math.isnan(E.stats[5]) else float("nan")

    rec_thrs = E.params.recThrs
    for k in (1, 2, 3):
        Ek = COCOeval(coco_gt, coco_dt, iouType="bbox")
        Ek.params.catIds = [k]
        Ek.evaluate(); Ek.accumulate()
        import io as _io
        import contextlib
        with contextlib.redirect_stdout(_io.StringIO()):
            Ek.summarize()
        ap50 = float(Ek.stats[1]); ap = float(Ek.stats[0])
        out["per_class"][k] = {
            "AP50": 0.0 if math.isnan(ap50) or ap50 < 0 else ap50,
            "AP5095": 0.0 if math.isnan(ap) or ap < 0 else ap,
        }
        try:
            prec = Ek.eval["precision"][0, :, 0, 0, 2]     # IoU=.50, all areas, maxDet=100
            prec = np.where(prec < 0, np.nan, prec)
            out["pr_curves"][k] = (rec_thrs.copy(), prec.copy())
        except Exception:
            out["pr_curves"][k] = (rec_thrs.copy(), np.full_like(rec_thrs, np.nan))
    return out


def prf_metrics(records, preds, score_thresh, iou_thr=0.5):
    """
    Explicit TP/FP/FN matching (greedy, score-descending, one GT per prediction).
    Gives precision / recall / F1 per class and overall (micro-averaged),
    plus a detection confusion matrix.
    """
    per_class = {k: {"TP": 0, "FP": 0, "FN": 0} for k in (1, 2, 3)}
    cm = np.zeros((4, 4), dtype=np.int64)    # rows GT (0=background), cols PRED
    per_image = {}

    for r in records:
        iid = r["image_id"]
        gtb, gtl = r["boxes"], r["labels"]
        p = preds.get(iid, {"boxes": np.zeros((0, 4), np.float32),
                            "scores": np.zeros((0,), np.float32),
                            "labels": np.zeros((0,), np.int64)})
        keep = p["scores"] >= score_thresh
        pb, ps, pl = p["boxes"][keep], p["scores"][keep], p["labels"][keep]
        order = np.argsort(-ps)
        pb, ps, pl = pb[order], ps[order], pl[order]

        matched_gt = np.zeros(len(gtb), dtype=bool)
        pred_is_tp = np.zeros(len(pb), dtype=bool)
        pred_match_gt = np.full(len(pb), -1, dtype=np.int64)

        if len(pb) and len(gtb):
            ious = box_iou_np(pb, gtb)
            for i in range(len(pb)):
                best, best_iou = -1, iou_thr
                for j in range(len(gtb)):
                    if matched_gt[j] or gtl[j] != pl[i]:
                        continue
                    if ious[i, j] >= best_iou:
                        best, best_iou = j, ious[i, j]
                if best >= 0:
                    matched_gt[best] = True
                    pred_is_tp[i] = True
                    pred_match_gt[i] = best

        for i in range(len(pb)):
            c = int(pl[i])
            if pred_is_tp[i]:
                per_class[c]["TP"] += 1
                cm[c, c] += 1
            else:
                per_class[c]["FP"] += 1
                # was it a class confusion or a pure background FP?
                if len(gtb):
                    ious_any = box_iou_np(pb[i:i + 1], gtb)[0]
                    j = int(np.argmax(ious_any)) if len(ious_any) else -1
                    if j >= 0 and ious_any[j] >= iou_thr:
                        cm[int(gtl[j]), c] += 1
                    else:
                        cm[0, c] += 1
                else:
                    cm[0, c] += 1

        for j in range(len(gtb)):
            if not matched_gt[j]:
                per_class[int(gtl[j])]["FN"] += 1
                cm[int(gtl[j]), 0] += 1

        per_image[iid] = {
            "n_gt": int(len(gtb)),
            "n_pred": int(len(pb)),
            "tp": int(pred_is_tp.sum()),
            "fp": int(len(pb) - pred_is_tp.sum()),
            "fn": int((~matched_gt).sum()),
            "pred_boxes": pb, "pred_scores": ps, "pred_labels": pl,
            "pred_is_tp": pred_is_tp, "matched_gt": matched_gt,
        }

    def prf(tp, fp, fn):
        p = tp / (tp + fp) if (tp + fp) else 0.0
        r = tp / (tp + fn) if (tp + fn) else 0.0
        f = 2 * p * r / (p + r) if (p + r) else 0.0
        return p, r, f

    res = {"per_class": {}, "confusion_matrix": cm, "per_image": per_image,
           "score_thresh": score_thresh, "iou_thr": iou_thr}
    TP = FP = FN = 0
    for k in (1, 2, 3):
        d = per_class[k]
        p, r, f = prf(d["TP"], d["FP"], d["FN"])
        res["per_class"][k] = {**d, "precision": p, "recall": r, "f1": f}
        TP += d["TP"]; FP += d["FP"]; FN += d["FN"]
    p, r, f = prf(TP, FP, FN)
    res["overall"] = {"TP": TP, "FP": FP, "FN": FN, "precision": p, "recall": r, "f1": f}
    res["macro"] = {
        "precision": float(np.mean([res["per_class"][k]["precision"] for k in (1, 2, 3)])),
        "recall": float(np.mean([res["per_class"][k]["recall"] for k in (1, 2, 3)])),
        "f1": float(np.mean([res["per_class"][k]["f1"] for k in (1, 2, 3)])),
    }
    return res


def evaluate_split(model, split_data, device, score_thresh, amp=True, desc="val",
                   preds=None):
    if preds is None:
        preds = run_inference(model, split_data["records"], device, amp=amp, desc=desc)
    coco_m = coco_eval_metrics(split_data["gt_dict"], preds, score_thresh=0.0)
    prf_m = prf_metrics(split_data["records"], preds, score_thresh)
    weakest = min(coco_m["per_class"][k]["AP50"] for k in (1, 2, 3))
    weakest_cls = min((1, 2, 3), key=lambda k: coco_m["per_class"][k]["AP50"])
    return {"preds": preds, "coco": coco_m, "prf": prf_m,
            "weakest_ap50": weakest, "weakest_class": weakest_cls}


def print_metric_table(coco_m, prf_m, title, names=CLASS_NAMES):
    hr(title, "-")
    log(f"  {'Class':<10}{'AP@50':>10}{'AP@50:95':>11}{'Precision':>11}{'Recall':>10}"
        f"{'F1':>9}{'TP':>7}{'FP':>7}{'FN':>7}")
    log("  " + "-" * 82)
    for k in (1, 2, 3):
        c = coco_m["per_class"][k]
        p = prf_m["per_class"][k]
        log(f"  {names[k]:<10}{100*c['AP50']:>9.2f}%{100*c['AP5095']:>10.2f}%"
            f"{100*p['precision']:>10.2f}%{100*p['recall']:>9.2f}%{100*p['f1']:>8.2f}%"
            f"{p['TP']:>7}{p['FP']:>7}{p['FN']:>7}")
    log("  " + "-" * 82)
    o = prf_m["overall"]
    log(f"  {'OVERALL':<10}{100*coco_m['mAP50']:>9.2f}%{100*coco_m['mAP5095']:>10.2f}%"
        f"{100*o['precision']:>10.2f}%{100*o['recall']:>9.2f}%{100*o['f1']:>8.2f}%"
        f"{o['TP']:>7}{o['FP']:>7}{o['FN']:>7}")
    log(f"  (precision/recall/F1 @ conf>={prf_m['score_thresh']:.2f}, IoU>={prf_m['iou_thr']:.2f};"
        f" AP is threshold-free)")
    log("")


def print_confusion(cm, names=CLASS_NAMES):
    hr("DETECTION CONFUSION MATRIX (rows = ground truth, cols = prediction)", "-")
    hdr = ["background"] + [names[k] for k in (1, 2, 3)]
    corner = "GT \\ PRED"
    log("  " + f"{corner:<12}" + "".join(f"{h:>12}" for h in hdr))
    for i, rn in enumerate(["background"] + [names[k] for k in (1, 2, 3)]):
        log(f"  {rn:<12}" + "".join(f"{cm[i, j]:>12}" for j in range(4)))
    log("  row 'background'  = false positives on background")
    log("  col 'background'  = missed ground-truth defects (false negatives)")
    log("  off-diagonal      = class confusion between defect types")
    log("")


# ==============================================================================
# 5. SANITY TEST
# ==============================================================================
def sanity_test(model, train_records, device, out_dir, n=8):
    import torch
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import matplotlib.patches as patches

    hr("SANITY TEST (must pass before full training)")
    ok = True
    os.makedirs(out_dir, exist_ok=True)

    with_ann = [r for r in train_records if len(r["labels"]) > 0]
    sample = random.sample(with_ann, min(n, len(with_ann)))
    log(f"  [1/6] loading {len(sample)} training images with ground truth ...")

    ds = FabricDetectionDataset(sample, train=True, aug_cfg=CONFIG["aug"])
    cols = min(4, len(sample))
    rows = int(math.ceil(len(sample) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
    axes = np.array(axes).reshape(-1)
    imgs, targets = [], []
    for i in range(len(sample)):
        img_t, tgt = ds[i]
        imgs.append(img_t)
        targets.append(tgt)
        ax = axes[i]
        ax.imshow(img_t.permute(1, 2, 0).numpy())
        for b, l in zip(tgt["boxes"].numpy(), tgt["labels"].numpy()):
            ax.add_patch(patches.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1],
                                           fill=False, lw=2,
                                           edgecolor=np.array(CLASS_COLORS[int(l)]) / 255.0))
            ax.text(b[0], max(0, b[1] - 4), CLASS_NAMES[int(l)], color="white", fontsize=8,
                    bbox=dict(facecolor=np.array(CLASS_COLORS[int(l)]) / 255.0,
                              alpha=0.85, pad=1, edgecolor="none"))
        ax.set_title(f"{sample[i]['file_name'][:26]}  ({len(tgt['labels'])} boxes)", fontsize=8)
        ax.axis("off")
    for j in range(len(sample), len(axes)):
        axes[j].axis("off")
    p = os.path.join(out_dir, "sanity_ground_truth.png")
    plt.tight_layout(); plt.savefig(p, dpi=110); plt.close()
    log(f"  [1/6] OK  -> ground-truth visualisation saved: {p}")

    log("  [2/6] verifying labels and boxes ...")
    for tgt in targets:
        b, l = tgt["boxes"], tgt["labels"]
        assert b.ndim == 2 and b.shape[1] == 4, "boxes must be (N,4)"
        if len(b):
            assert bool((b[:, 2] > b[:, 0]).all() and (b[:, 3] > b[:, 1]).all()), "x2>x1, y2>y1 violated"
            assert int(l.min()) >= 1 and int(l.max()) <= 3, f"labels out of range: {l}"
        assert l.dtype == torch.int64, "labels must be int64"
    log("  [2/6] OK  - all labels in [1,3], all boxes have positive width/height")

    log("  [3/6] one training forward/backward step ...")
    model.train()
    bs = min(2, len(imgs))
    im = [i.to(device) for i in imgs[:bs]]
    tg = [{k: v.to(device) for k, v in t.items()} for t in targets[:bs]]
    loss_dict = model(im, tg)
    losses = sum(loss_dict.values())
    log("        loss components:")
    for k, v in loss_dict.items():
        log(f"          {k:<22} {float(v):.5f}")
    log(f"        total loss             {float(losses):.5f}")
    if not math.isfinite(float(losses)):
        ok = False
        log("  [3/6] FAIL - loss is not finite")
    else:
        losses.backward()
        gn = 0.0
        for prm in model.parameters():
            if prm.grad is not None:
                gn += float(prm.grad.detach().norm()) ** 2
        gn = gn ** 0.5
        model.zero_grad(set_to_none=True)
        log(f"  [3/6] OK  - loss finite, backward ran, grad-norm = {gn:.3f}")
        if not math.isfinite(gn):
            ok = False
            log("  [3/6] FAIL - gradient norm not finite")

    log("  [4/6] verifying empty-target (negative image) handling ...")
    try:
        empty_t = [{"boxes": torch.zeros((0, 4), dtype=torch.float32, device=device),
                    "labels": torch.zeros((0,), dtype=torch.int64, device=device)}]
        ld = model([im[0]], empty_t)
        v = float(sum(ld.values()))
        model.zero_grad(set_to_none=True)
        log(f"  [4/6] OK  - background-only image gives finite loss ({v:.5f})")
    except Exception as e:
        log(f"  [4/6] WARN - empty target failed: {e}")

    log("  [5/6] inference output shapes ...")
    model.eval()
    with torch.no_grad():
        out = model([im[0]])
    o = out[0]
    log(f"        keys        : {sorted(o.keys())}")
    log(f"        boxes shape : {tuple(o['boxes'].shape)}   (expect (N,4))")
    log(f"        scores shape: {tuple(o['scores'].shape)}")
    log(f"        labels shape: {tuple(o['labels'].shape)}")
    if len(o["labels"]):
        log(f"        label range : {int(o['labels'].min())}..{int(o['labels'].max())}  (expect 1..3)")
        if int(o["labels"].max()) > 3 or int(o["labels"].min()) < 1:
            ok = False
            log("  [5/6] FAIL - predicted labels outside 1..3")
    if o["boxes"].ndim != 2 or o["boxes"].shape[1] != 4:
        ok = False
        log("  [5/6] FAIL - bad box shape")
    else:
        log("  [5/6] OK")

    log("  [6/6] classifier head width ...")
    nc = model.roi_heads.box_predictor.cls_score.out_features
    log(f"        cls_score.out_features = {nc}")
    if nc != CONFIG["num_classes"]:
        ok = False
        log(f"  [6/6] FAIL - must be {CONFIG['num_classes']} "
            f"(background + {CONFIG['num_classes']-1} classes)")
    else:
        log("  [6/6] OK")

    model.zero_grad(set_to_none=True)
    if device.type == "cuda":
        import torch as _t
        _t.cuda.empty_cache()
    hr("SANITY TEST: " + ("PASSED" if ok else "FAILED"))
    log("")
    return ok


# ==============================================================================
# 6. TRAINING
# ==============================================================================
def selection_score(map50, weakest_ap50, map5095):
    return (CONFIG["sel_w_map50"] * map50 +
            CONFIG["sel_w_weakest"] * weakest_ap50 +
            CONFIG["sel_w_map5095"] * map5095)


def train_one_experiment(exp, data, device, gpu_mem_gb, out_root, t_start):
    """Trains one configuration and returns its full history + best state path."""
    import torch

    exp_dir = os.path.join(out_root, "experiments", exp["name"])
    ckpt_dir = os.path.join(exp_dir, "checkpoints")
    os.makedirs(ckpt_dir, exist_ok=True)

    hr(f"EXPERIMENT {exp['name'].upper()}  |  min_size={exp['min_size']} "
       f"max_size={exp['max_size']} lr={exp['lr']:g}")

    seed_everything(CONFIG["seed"])

    bs = CONFIG["batch_size"] or auto_batch_size(gpu_mem_gb, exp["min_size"])
    accum = CONFIG["grad_accum_steps"] or max(1, int(math.ceil(4 / bs)))
    log(f"  batch_size       : {bs}  (auto from {gpu_mem_gb:.1f} GB GPU memory)")
    log(f"  grad accumulation: {accum}  -> effective batch {bs*accum}")
    log(f"  num_workers      : {auto_num_workers()}   pin_memory={device.type=='cuda'}")

    model = build_model(CONFIG["num_classes"], exp["min_size"], exp["max_size"], device)
    optimizer = build_optimizer(model, exp["lr"], CONFIG["weight_decay"])

    train_records = data["train"]["records"]
    if CONFIG["debug_subset"]:
        train_records = train_records[:CONFIG["debug_subset"]]
        log(f"  [debug] using only {len(train_records)} training images")

    train_loader = make_loader(train_records, True, bs, CONFIG["aug"], device)
    steps_per_epoch = max(1, len(train_loader) // accum)
    scheduler = build_scheduler(optimizer, steps_per_epoch, CONFIG["epochs"])

    use_amp = CONFIG["amp"] and device.type == "cuda"
    try:
        scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
        autocast_ctx = lambda: torch.amp.autocast("cuda", enabled=use_amp)
    except Exception:
        scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
        autocast_ctx = lambda: torch.cuda.amp.autocast(enabled=use_amp)
    log(f"  mixed precision  : {'AMP enabled' if use_amp else 'disabled (CPU/float32)'}")
    log(f"  gradient clipping: max_norm={CONFIG['grad_clip_norm']}")
    log(f"  epochs           : up to {CONFIG['epochs']}, early stop patience "
        f"{CONFIG['early_stop_patience']}")
    log("")

    history = []
    best = {"score": -1.0, "map50": -1.0, "map5095": -1.0, "weakest": -1.0,
            "epoch": -1, "f1": -1.0}
    best_map50_only = {"map50": -1.0, "epoch": -1}
    start_epoch = 0
    epochs_no_improve = 0

    # ---------- resume ----------
    last_path = os.path.join(exp_dir, "last_model.pth")
    if os.path.isfile(last_path):
        try:
            ck = torch.load(last_path, map_location=device, weights_only=False)
            model.load_state_dict(ck["model"])
            optimizer.load_state_dict(ck["optimizer"])
            scheduler.load_state_dict(ck["scheduler"])
            if ck.get("scaler") and use_amp:
                scaler.load_state_dict(ck["scaler"])
            start_epoch = ck["epoch"] + 1
            history = ck.get("history", [])
            best = ck.get("best", best)
            best_map50_only = ck.get("best_map50_only", best_map50_only)
            epochs_no_improve = ck.get("epochs_no_improve", 0)
            log(f"  >>> RESUMED from {last_path} at epoch {start_epoch} "
                f"(best val mAP@50 so far {100*best['map50']:.2f}%)")
        except Exception as e:
            log(f"  [warn] could not resume ({e}); starting from scratch")

    stop_reason = "completed all epochs"

    for epoch in range(start_epoch, CONFIG["epochs"]):
        # ---------------- train ----------------
        model.train()
        t_ep = time.time()
        running = defaultdict(float)
        n_batches = 0
        optimizer.zero_grad(set_to_none=True)
        skipped = 0

        for it, (imgs, targets) in enumerate(train_loader):
            imgs = [i.to(device, non_blocking=True) for i in imgs]
            targets = [{k: v.to(device, non_blocking=True) for k, v in t.items()}
                       for t in targets]
            try:
                with autocast_ctx():
                    loss_dict = model(imgs, targets)
                    loss = sum(loss_dict.values())
            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    log(f"    [OOM] batch {it} skipped; clearing cache")
                    optimizer.zero_grad(set_to_none=True)
                    if device.type == "cuda":
                        torch.cuda.empty_cache()
                    skipped += 1
                    continue
                raise

            lv = float(loss.detach())
            if not math.isfinite(lv):
                log(f"    [warn] non-finite loss at iter {it} - batch skipped")
                optimizer.zero_grad(set_to_none=True)
                skipped += 1
                continue

            scaler.scale(loss / accum).backward()

            if (it + 1) % accum == 0 or (it + 1) == len(train_loader):
                if CONFIG["grad_clip_norm"]:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip_norm"])
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

            running["loss"] += lv
            for k, v in loss_dict.items():
                running[k] += float(v.detach())
            n_batches += 1

            if it % max(1, len(train_loader) // 8) == 0:
                log(f"    ep {epoch+1:>3}/{CONFIG['epochs']}  it {it:>5}/{len(train_loader)}  "
                    f"loss {lv:.4f}  lr {optimizer.param_groups[1]['lr']:.2e}  "
                    f"elapsed {fmt_hms(time.time()-t_start)}")

        n_batches = max(1, n_batches)
        train_losses = {k: v / n_batches for k, v in running.items()}
        train_time = time.time() - t_ep

        # ---------------- validate ----------------
        val = evaluate_split(model, data["valid"], device, 0.50,
                             amp=use_amp, desc=f"val ep{epoch+1}")
        cm_c, cm_p = val["coco"], val["prf"]
        map50, map5095 = cm_c["mAP50"], cm_c["mAP5095"]
        weakest, weak_cls = val["weakest_ap50"], val["weakest_class"]
        f1 = cm_p["overall"]["f1"]

        row = {
            "experiment": exp["name"], "epoch": epoch + 1,
            "train_loss": train_losses["loss"],
            "loss_classifier": train_losses.get("loss_classifier", float("nan")),
            "loss_box_reg": train_losses.get("loss_box_reg", float("nan")),
            "loss_objectness": train_losses.get("loss_objectness", float("nan")),
            "loss_rpn_box_reg": train_losses.get("loss_rpn_box_reg", float("nan")),
            "lr_head": optimizer.param_groups[1]["lr"],
            "lr_backbone": optimizer.param_groups[0]["lr"],
            "val_mAP50": map50, "val_mAP5095": map5095,
            "val_precision": cm_p["overall"]["precision"],
            "val_recall": cm_p["overall"]["recall"],
            "val_f1": f1,
            "val_weakest_class": CLASS_NAMES[weak_cls],
            "val_weakest_AP50": weakest,
            "train_time_s": train_time,
            "skipped_batches": skipped,
        }
        for k in (1, 2, 3):
            row[f"AP50_{CLASS_NAMES[k]}"] = cm_c["per_class"][k]["AP50"]
            row[f"AP5095_{CLASS_NAMES[k]}"] = cm_c["per_class"][k]["AP5095"]
            row[f"P_{CLASS_NAMES[k]}"] = cm_p["per_class"][k]["precision"]
            row[f"R_{CLASS_NAMES[k]}"] = cm_p["per_class"][k]["recall"]
            row[f"F1_{CLASS_NAMES[k]}"] = cm_p["per_class"][k]["f1"]
        history.append(row)

        log("")
        hr(f"EPOCH {epoch+1}/{CONFIG['epochs']}  [{exp['name']}]  "
           f"train {fmt_hms(train_time)}  total {fmt_hms(time.time()-t_start)}", "-")
        log(f"  train loss {train_losses['loss']:.4f}  "
            f"(cls {train_losses.get('loss_classifier',0):.4f} | "
            f"box {train_losses.get('loss_box_reg',0):.4f} | "
            f"obj {train_losses.get('loss_objectness',0):.4f} | "
            f"rpn {train_losses.get('loss_rpn_box_reg',0):.4f})")
        print_metric_table(cm_c, cm_p, f"VALIDATION  epoch {epoch+1}")
        log(f"  weakest class: {CLASS_NAMES[weak_cls]}  AP@50 = {100*weakest:.2f}%")

        # ---------------- checkpoint selection ----------------
        score = selection_score(map50, weakest, map5095)
        collapse = (best["weakest"] > 0 and weakest < CONFIG["collapse_guard_ratio"] * best["weakest"])
        improved = (score > best["score"] + 1e-6) and not collapse

        if map50 > best_map50_only["map50"]:
            best_map50_only = {"map50": map50, "epoch": epoch + 1}

        if collapse and score > best["score"]:
            log(f"  [selection] REJECTED: overall score improved but weakest-class AP@50 "
                f"collapsed ({100*weakest:.2f}% < {100*CONFIG['collapse_guard_ratio']*best['weakest']:.2f}%)")

        state = {
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict() if use_amp else None,
            "epoch": epoch,
            "history": history,
            "best": best,
            "best_map50_only": best_map50_only,
            "epochs_no_improve": epochs_no_improve,
            "config": {**{k: v for k, v in CONFIG.items() if k != "aug"},
                       "aug": CONFIG["aug"], "experiment": exp},
            "class_names": CLASS_NAMES,
        }

        if improved:
            best = {"score": score, "map50": map50, "map5095": map5095,
                    "weakest": weakest, "epoch": epoch + 1, "f1": f1,
                    "weakest_class": CLASS_NAMES[weak_cls],
                    "precision": cm_p["overall"]["precision"],
                    "recall": cm_p["overall"]["recall"]}
            state["best"] = best
            epochs_no_improve = 0
            state["epochs_no_improve"] = 0
            torch.save(state, os.path.join(exp_dir, "best_model.pth"))
            log(f"  [selection] NEW BEST (score {score:.4f}) -> best_model.pth saved")
            log(f"              mAP@50 {100*map50:.2f}% | mAP@50:95 {100*map5095:.2f}% | "
                f"weakest {100*weakest:.2f}% | F1 {100*f1:.2f}%")
        else:
            epochs_no_improve += 1
            state["epochs_no_improve"] = epochs_no_improve
            log(f"  [selection] no improvement ({epochs_no_improve}/"
                f"{CONFIG['early_stop_patience']})  best score {best['score']:.4f} "
                f"@ epoch {best['epoch']}")

        torch.save(state, last_path)
        if (epoch + 1) % CONFIG["checkpoint_every"] == 0:
            torch.save(state, os.path.join(ckpt_dir, f"epoch_{epoch+1:03d}.pth"))

        save_metrics_csv(history, os.path.join(out_root, "metrics"), exp["name"])
        log("")

        # ---------------- stopping ----------------
        if epochs_no_improve >= CONFIG["early_stop_patience"]:
            stop_reason = (f"early stopping - no improvement for "
                           f"{CONFIG['early_stop_patience']} epochs")
            log(f"  >>> {stop_reason}")
            break
        if (time.time() - t_start) / 3600.0 > CONFIG["max_train_hours"]:
            stop_reason = f"time budget reached ({CONFIG['max_train_hours']}h)"
            log(f"  >>> {stop_reason}. Re-run the script to resume from last_model.pth.")
            break

    del model
    if device.type == "cuda":
        import torch as _t
        _t.cuda.empty_cache()

    return {"name": exp["name"], "exp": exp, "dir": exp_dir, "history": history,
            "best": best, "best_map50_only": best_map50_only,
            "best_path": os.path.join(exp_dir, "best_model.pth"),
            "last_path": last_path, "batch_size": bs, "grad_accum": accum,
            "stop_reason": stop_reason}


# ==============================================================================
# 7. OUTPUT: csv, plots, visualisation, error analysis
# ==============================================================================
def save_metrics_csv(history, metrics_dir, exp_name):
    import csv
    os.makedirs(metrics_dir, exist_ok=True)
    if not history:
        return
    path = os.path.join(metrics_dir, "metrics.csv")
    rows = []
    if os.path.isfile(path):
        with open(path, newline="") as f:
            rows = [r for r in csv.DictReader(f) if r.get("experiment") != exp_name]
    keys = list(history[0].keys())
    all_keys = list(dict.fromkeys(keys + [k for r in rows for k in r.keys()]))
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=all_keys)
        w.writeheader()
        for r in rows:
            w.writerow(r)
        for r in history:
            w.writerow(r)


def save_per_class_csv(coco_m, prf_m, metrics_dir, tag):
    import csv
    os.makedirs(metrics_dir, exist_ok=True)
    path = os.path.join(metrics_dir, "per_class_metrics.csv")
    exists = os.path.isfile(path)
    with open(path, "a", newline="") as f:
        w = csv.writer(f)
        if not exists:
            w.writerow(["evaluation", "class", "AP@50", "AP@50:95", "precision",
                        "recall", "f1", "TP", "FP", "FN", "conf_threshold"])
        for k in (1, 2, 3):
            c, p = coco_m["per_class"][k], prf_m["per_class"][k]
            w.writerow([tag, CLASS_NAMES[k], f"{c['AP50']:.6f}", f"{c['AP5095']:.6f}",
                        f"{p['precision']:.6f}", f"{p['recall']:.6f}", f"{p['f1']:.6f}",
                        p["TP"], p["FP"], p["FN"], f"{prf_m['score_thresh']:.2f}"])
        o = prf_m["overall"]
        w.writerow([tag, "OVERALL", f"{coco_m['mAP50']:.6f}", f"{coco_m['mAP5095']:.6f}",
                    f"{o['precision']:.6f}", f"{o['recall']:.6f}", f"{o['f1']:.6f}",
                    o["TP"], o["FP"], o["FN"], f"{prf_m['score_thresh']:.2f}"])


def plot_training_curves(all_hist, plots_dir):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    os.makedirs(plots_dir, exist_ok=True)

    # --- loss ---
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    for name, hist in all_hist.items():
        ep = [h["epoch"] for h in hist]
        ax[0].plot(ep, [h["train_loss"] for h in hist], marker="o", ms=3, label=f"{name} total")
    ax[0].set_xlabel("epoch"); ax[0].set_ylabel("training loss")
    ax[0].set_title("Training loss"); ax[0].grid(alpha=.3); ax[0].legend()

    name0 = list(all_hist.keys())[0]
    h0 = all_hist[name0]
    ep = [h["epoch"] for h in h0]
    for comp in ["loss_classifier", "loss_box_reg", "loss_objectness", "loss_rpn_box_reg"]:
        ax[1].plot(ep, [h.get(comp, np.nan) for h in h0], marker=".", ms=3, label=comp)
    ax[1].set_xlabel("epoch"); ax[1].set_ylabel("loss"); ax[1].grid(alpha=.3)
    ax[1].set_title(f"Loss components ({name0})"); ax[1].legend(fontsize=8)
    plt.tight_layout(); plt.savefig(os.path.join(plots_dir, "training_loss.png"), dpi=130)
    plt.close()

    # --- validation metrics ---
    fig, ax = plt.subplots(2, 2, figsize=(14, 9))
    for name, hist in all_hist.items():
        ep = [h["epoch"] for h in hist]
        ax[0, 0].plot(ep, [100 * h["val_mAP50"] for h in hist], marker="o", ms=3, label=f"{name} mAP@50")
        ax[0, 0].plot(ep, [100 * h["val_mAP5095"] for h in hist], marker="s", ms=3,
                      ls="--", label=f"{name} mAP@50:95")
        ax[0, 1].plot(ep, [100 * h["val_f1"] for h in hist], marker="o", ms=3, label=f"{name} F1")
        ax[0, 1].plot(ep, [100 * h["val_precision"] for h in hist], ls="--", label=f"{name} precision")
        ax[0, 1].plot(ep, [100 * h["val_recall"] for h in hist], ls=":", label=f"{name} recall")
        ax[1, 1].plot(ep, [100 * h["val_weakest_AP50"] for h in hist], marker="o", ms=3,
                      label=f"{name} weakest-class AP@50")
    ax[0, 0].set_title("Validation mAP"); ax[0, 0].set_ylabel("%")
    ax[0, 1].set_title("Validation precision / recall / F1 @ conf 0.50")
    ep0 = [h["epoch"] for h in h0]
    for k in (1, 2, 3):
        ax[1, 0].plot(ep0, [100 * h[f"AP50_{CLASS_NAMES[k]}"] for h in h0],
                      marker="o", ms=3, label=CLASS_NAMES[k],
                      color=np.array(CLASS_COLORS[k]) / 255.0)
    ax[1, 0].set_title(f"Per-class validation AP@50 ({name0})")
    ax[1, 1].set_title("Weakest-class AP@50 (collapse monitor)")
    for a in ax.reshape(-1):
        a.set_xlabel("epoch"); a.grid(alpha=.3); a.legend(fontsize=8)
    plt.tight_layout(); plt.savefig(os.path.join(plots_dir, "validation_map.png"), dpi=130)
    plt.close()


def plot_pr_curves(coco_m, plots_dir, tag):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    os.makedirs(plots_dir, exist_ok=True)
    plt.figure(figsize=(7.5, 6))
    for k in (1, 2, 3):
        if k not in coco_m["pr_curves"]:
            continue
        rec, prec = coco_m["pr_curves"][k]
        plt.plot(rec, prec, lw=2, color=np.array(CLASS_COLORS[k]) / 255.0,
                 label=f"{CLASS_NAMES[k]}  AP@50={100*coco_m['per_class'][k]['AP50']:.1f}%")
    plt.xlabel("Recall"); plt.ylabel("Precision")
    plt.title(f"Precision-Recall @ IoU=0.50  ({tag})\nmAP@50 = {100*coco_m['mAP50']:.2f}%")
    plt.xlim(0, 1); plt.ylim(0, 1.02); plt.grid(alpha=.3); plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, f"precision_recall_{tag}.png"), dpi=130)
    if tag.startswith("test"):
        plt.savefig(os.path.join(plots_dir, "precision_recall.png"), dpi=130)
    plt.close()


def plot_threshold_sweep(sweep, plots_dir):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    os.makedirs(plots_dir, exist_ok=True)
    th = [s["threshold"] for s in sweep]
    plt.figure(figsize=(9, 6))
    plt.plot(th, [100 * s["precision"] for s in sweep], marker="o", label="precision")
    plt.plot(th, [100 * s["recall"] for s in sweep], marker="s", label="recall")
    plt.plot(th, [100 * s["f1"] for s in sweep], marker="^", lw=2.5, label="F1")
    for k in (1, 2, 3):
        plt.plot(th, [100 * s[f"f1_{CLASS_NAMES[k]}"] for s in sweep], ls="--", lw=1,
                 color=np.array(CLASS_COLORS[k]) / 255.0, label=f"F1 {CLASS_NAMES[k]}")
    best = max(sweep, key=lambda s: s["f1"])
    plt.axvline(best["threshold"], color="k", ls=":", alpha=.6)
    plt.title(f"Confidence threshold sweep (VALIDATION only)\nbest F1 at "
              f"conf={best['threshold']:.2f}")
    plt.xlabel("confidence threshold"); plt.ylabel("%")
    plt.grid(alpha=.3); plt.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, "confidence_threshold_sweep.png"), dpi=130)
    plt.close()


def plot_confusion(cm, plots_dir, tag):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    labels = ["background"] + [CLASS_NAMES[k] for k in (1, 2, 3)]
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(4)); ax.set_xticklabels(labels, rotation=40, ha="right")
    ax.set_yticks(range(4)); ax.set_yticklabels(labels)
    ax.set_xlabel("predicted"); ax.set_ylabel("ground truth")
    ax.set_title(f"Detection confusion matrix ({tag})")
    mx = cm.max() if cm.max() else 1
    for i in range(4):
        for j in range(4):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > 0.55 * mx else "black", fontsize=10)
    fig.colorbar(im)
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, f"confusion_matrix_{tag}.png"), dpi=130)
    plt.close()


def draw_and_save(record, prf_img, out_path, title_extra=""):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import matplotlib.patches as patches
    import cv2

    img = cv2.imread(record["path"], cv2.IMREAD_COLOR)
    if img is None:
        return False
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    fig, ax = plt.subplots(figsize=(11, 8))
    ax.imshow(img)

    for b, l in zip(record["boxes"], record["labels"]):
        ax.add_patch(patches.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1],
                                       fill=False, lw=2.2, edgecolor="#00ff66", ls="-"))
        ax.text(b[0], max(0, b[1] - 3), f"GT {CLASS_NAMES[int(l)]}", fontsize=8,
                color="black", bbox=dict(facecolor="#00ff66", alpha=.85, pad=1, edgecolor="none"))

    for b, s, l, tp in zip(prf_img["pred_boxes"], prf_img["pred_scores"],
                           prf_img["pred_labels"], prf_img["pred_is_tp"]):
        col = "#ffcc00" if tp else "#ff2d2d"
        ax.add_patch(patches.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1],
                                       fill=False, lw=2.0, edgecolor=col, ls="--"))
        ax.text(b[2], min(img.shape[0] - 2, b[3] + 11),
                f"{CLASS_NAMES[int(l)]} {s:.2f}{'' if tp else ' FP'}",
                fontsize=8, ha="right", color="black",
                bbox=dict(facecolor=col, alpha=.85, pad=1, edgecolor="none"))

    ax.set_title(f"{record['file_name']}  |  GT={prf_img['n_gt']}  pred={prf_img['n_pred']}  "
                 f"TP={prf_img['tp']} FP={prf_img['fp']} FN={prf_img['fn']}  {title_extra}",
                 fontsize=10)
    ax.axis("off")
    plt.tight_layout()
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    plt.savefig(out_path, dpi=110, bbox_inches="tight")
    plt.close()
    return True


def save_visualisations(split_data, prf_m, out_dir, n=20, tag="validation"):
    """Deliberately picks a mix: clean hits, misses, FPs, crowded, small, hard."""
    os.makedirs(out_dir, exist_ok=True)
    recs = {r["image_id"]: r for r in split_data["records"]}
    pi = prf_m["per_image"]

    def small_obj(r):
        if not len(r["boxes"]):
            return 0
        a = (r["boxes"][:, 2] - r["boxes"][:, 0]) * (r["boxes"][:, 3] - r["boxes"][:, 1])
        return int((a < 32 * 32).sum())

    ids = list(pi.keys())
    buckets = {
        "correct": sorted([i for i in ids if pi[i]["tp"] > 0 and pi[i]["fp"] == 0 and pi[i]["fn"] == 0],
                          key=lambda i: -pi[i]["tp"]),
        "missed": sorted([i for i in ids if pi[i]["fn"] > 0], key=lambda i: -pi[i]["fn"]),
        "false_positive": sorted([i for i in ids if pi[i]["fp"] > 0], key=lambda i: -pi[i]["fp"]),
        "multiple": sorted([i for i in ids if pi[i]["n_gt"] >= 3], key=lambda i: -pi[i]["n_gt"]),
        "small": sorted([i for i in ids if small_obj(recs[i]) > 0],
                        key=lambda i: -small_obj(recs[i])),
        "mixed_hard": sorted([i for i in ids if pi[i]["fp"] > 0 and pi[i]["fn"] > 0],
                             key=lambda i: -(pi[i]["fp"] + pi[i]["fn"])),
    }
    quota = max(1, n // len(buckets))
    chosen, seen = [], set()
    for bname, lst in buckets.items():
        for i in lst:
            if i in seen:
                continue
            chosen.append((bname, i)); seen.add(i)
            if len([c for c in chosen if c[0] == bname]) >= quota:
                break
    for i in ids:
        if len(chosen) >= n:
            break
        if i not in seen:
            chosen.append(("other", i)); seen.add(i)

    saved = 0
    for bname, iid in chosen[:n]:
        p = os.path.join(out_dir, f"{saved:02d}_{bname}_{recs[iid]['file_name']}.png")
        if draw_and_save(recs[iid], pi[iid], p, title_extra=f"[{bname}]"):
            saved += 1
    log(f"  saved {saved} {tag} prediction images -> {out_dir}")
    return saved


def save_error_analysis(split_data, prf_m, out_root, n_per_class=20, tag="validation"):
    """Per-class FP and FN galleries + a written breakdown."""
    fp_dir = os.path.join(out_root, "error_analysis", "false_positives")
    fn_dir = os.path.join(out_root, "error_analysis", "false_negatives")
    os.makedirs(fp_dir, exist_ok=True)
    os.makedirs(fn_dir, exist_ok=True)

    recs = {r["image_id"]: r for r in split_data["records"]}
    pi = prf_m["per_image"]

    fp_by_class = defaultdict(list)
    fn_by_class = defaultdict(list)
    fp_conf = defaultdict(list)
    fp_confusion = defaultdict(Counter)
    fn_size = defaultdict(list)
    fn_edge = Counter()

    for iid, d in pi.items():
        r = recs[iid]
        for i, (b, s, l, tp) in enumerate(zip(d["pred_boxes"], d["pred_scores"],
                                              d["pred_labels"], d["pred_is_tp"])):
            if tp:
                continue
            c = int(l)
            fp_by_class[c].append((iid, float(s)))
            fp_conf[c].append(float(s))
            if len(r["boxes"]):
                ious = box_iou_np(b[None], r["boxes"])[0]
                j = int(np.argmax(ious))
                if ious[j] >= 0.3:
                    fp_confusion[c][CLASS_NAMES[int(r["labels"][j])]] += 1
                else:
                    fp_confusion[c]["background"] += 1
            else:
                fp_confusion[c]["background(empty image)"] += 1
        for j, m in enumerate(d["matched_gt"]):
            if m:
                continue
            c = int(r["labels"][j])
            fn_by_class[c].append(iid)
            b = r["boxes"][j]
            fn_size[c].append(float((b[2] - b[0]) * (b[3] - b[1])))
            if b[0] <= 2 or b[1] <= 2 or b[2] >= r["width"] - 2 or b[3] >= r["height"] - 2:
                fn_edge[c] += 1

    for c in (1, 2, 3):
        seen = set()
        for iid, s in sorted(fp_by_class[c], key=lambda x: -x[1]):
            if iid in seen:
                continue
            seen.add(iid)
            draw_and_save(recs[iid], pi[iid],
                          os.path.join(fp_dir, f"{CLASS_NAMES[c]}_{len(seen):02d}_"
                                               f"conf{s:.2f}_{recs[iid]['file_name']}.png"),
                          title_extra=f"[FP {CLASS_NAMES[c]}]")
            if len(seen) >= n_per_class:
                break
        seen = set()
        for iid in fn_by_class[c]:
            if iid in seen:
                continue
            seen.add(iid)
            draw_and_save(recs[iid], pi[iid],
                          os.path.join(fn_dir, f"{CLASS_NAMES[c]}_{len(seen):02d}_"
                                               f"{recs[iid]['file_name']}.png"),
                          title_extra=f"[FN {CLASS_NAMES[c]}]")
            if len(seen) >= n_per_class:
                break

    hr(f"ERROR ANALYSIS ({tag}, conf>={prf_m['score_thresh']:.2f}, IoU>={prf_m['iou_thr']:.2f})", "-")
    notes = {
        1: "HOLE  - check small and edge-touching defects.",
        2: "KNOT  - check confusion with normal fabric texture.",
        3: "STAIN - texture/lighting variation makes boundaries ambiguous; IoU often "
           "just under 0.5 rather than a true miss.",
    }
    for c in (1, 2, 3):
        d = prf_m["per_class"][c]
        log(f"  {CLASS_NAMES[c].upper():<6} TP={d['TP']:<5} FP={d['FP']:<5} FN={d['FN']:<5} "
            f"P={100*d['precision']:.1f}%  R={100*d['recall']:.1f}%  F1={100*d['f1']:.1f}%")
        log(f"         {notes[c]}")
        if fp_conf[c]:
            a = np.array(fp_conf[c])
            log(f"         FP confidence: mean={a.mean():.3f} median={np.median(a):.3f} "
                f"max={a.max():.3f} | {int((a>0.5).sum())} of {len(a)} above 0.50")
            log(f"         FP overlaps  : {dict(fp_confusion[c].most_common(5))}")
        if fn_size[c]:
            a = np.array(fn_size[c])
            log(f"         FN box area  : p25={np.percentile(a,25):.0f}px "
                f"median={np.median(a):.0f}px p75={np.percentile(a,75):.0f}px | "
                f"{int((a < 32*32).sum())} of {len(a)} are smaller than 32x32")
            log(f"         FN touching image edge: {fn_edge[c]}")
        log("")
    log(f"  FP galleries -> {fp_dir}")
    log(f"  FN galleries -> {fn_dir}")
    log("")

    return {"fp_confusion": {CLASS_NAMES[c]: dict(fp_confusion[c]) for c in (1, 2, 3)},
            "fn_edge": {CLASS_NAMES[c]: fn_edge[c] for c in (1, 2, 3)}}


# ==============================================================================
# 8. CONFIDENCE THRESHOLD SWEEP (VALIDATION ONLY)
# ==============================================================================
def sweep_confidence(val_data, val_preds, out_root):
    hr("CONFIDENCE THRESHOLD SWEEP  --  VALIDATION SET ONLY")
    log("  (the test set is NOT used anywhere in this decision)")
    log("")
    log(f"  {'conf':>6}{'Precision':>11}{'Recall':>10}{'F1':>9}{'TP':>8}{'FP':>8}{'FN':>8}"
        f"{'F1 hole':>10}{'F1 knot':>10}{'F1 stain':>10}")
    log("  " + "-" * 100)
    sweep = []
    for t in CONFIG["conf_thresholds"]:
        m = prf_metrics(val_data["records"], val_preds, t, CONFIG["iou_for_prf"])
        o = m["overall"]
        row = {"threshold": t, "precision": o["precision"], "recall": o["recall"],
               "f1": o["f1"], "TP": o["TP"], "FP": o["FP"], "FN": o["FN"],
               "macro_f1": m["macro"]["f1"],
               "min_class_f1": min(m["per_class"][k]["f1"] for k in (1, 2, 3))}
        for k in (1, 2, 3):
            row[f"f1_{CLASS_NAMES[k]}"] = m["per_class"][k]["f1"]
            row[f"p_{CLASS_NAMES[k]}"] = m["per_class"][k]["precision"]
            row[f"r_{CLASS_NAMES[k]}"] = m["per_class"][k]["recall"]
        sweep.append(row)
        log(f"  {t:>6.2f}{100*o['precision']:>10.2f}%{100*o['recall']:>9.2f}%"
            f"{100*o['f1']:>8.2f}%{o['TP']:>8}{o['FP']:>8}{o['FN']:>8}"
            + "".join(f"{100*row[f'f1_{CLASS_NAMES[k]}']:>9.2f}%" for k in (1, 2, 3)))
    log("")

    # choose: maximise overall F1, but require the weakest class not to be sacrificed
    best_f1 = max(s["f1"] for s in sweep)
    cands = [s for s in sweep if s["f1"] >= best_f1 - 0.005]
    best = max(cands, key=lambda s: (s["min_class_f1"], s["f1"]))
    log(f"  SELECTED confidence threshold = {best['threshold']:.2f}")
    log(f"    overall F1 {100*best['f1']:.2f}%  precision {100*best['precision']:.2f}%  "
        f"recall {100*best['recall']:.2f}%")
    log(f"    weakest per-class F1 at this threshold: {100*best['min_class_f1']:.2f}%")
    log("    (among thresholds within 0.5pp of the best overall F1, the one with the "
        "strongest weakest-class F1 is chosen)")
    log("")

    import csv
    os.makedirs(os.path.join(out_root, "metrics"), exist_ok=True)
    with open(os.path.join(out_root, "metrics", "confidence_sweep_validation.csv"),
              "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(sweep[0].keys()))
        w.writeheader()
        for r in sweep:
            w.writerow(r)
    plot_threshold_sweep(sweep, os.path.join(out_root, "plots"))
    return best["threshold"], sweep


# ==============================================================================
# 9. FINAL REPORT
# ==============================================================================
def write_final_report(path, ctx):
    L = []
    A = L.append
    A("=" * 78)
    A(" FABRIC DEFECT DETECTION - FINAL REPORT")
    A(" Faster R-CNN + ResNet-50 FPN (torchvision)")
    A("=" * 78)
    A(f" generated              : {time.strftime('%Y-%m-%d %H:%M:%S')}")
    A(f" total wall-clock time  : {fmt_hms(ctx['total_time'])}")
    A("")
    A("-" * 78)
    A(" ENVIRONMENT")
    A("-" * 78)
    for k, v in ctx["env"].items():
        A(f"   {k:<22}: {v}")
    A("")
    A("-" * 78)
    A(" DATASET (as auto-detected; original files never modified)")
    A("-" * 78)
    A(f"   root                  : {ctx['root']}")
    for s in ("train", "valid", "test"):
        if s not in ctx["data"]:
            A(f"   {s:<22}: NOT PRESENT")
            continue
        d = ctx["data"][s]
        dist = d["class_distribution"]
        A(f"   {s:<22}: {len(d['records'])} images, "
          f"{len(d['gt_dict']['annotations'])} annotations")
        A(f"   {'':<22}  json = {ctx['splits'][s]['json']}")
        A(f"   {'':<22}  imgs = {ctx['splits'][s]['images_dir']}")
        A(f"   {'':<22}  hole={dist.get(1,0)} knot={dist.get(2,0)} "
          f"stain={dist.get(3,0)}")
        A(f"   {'':<22}  cleaning: {d['clean_stats']}")
    A("")
    A("   label mapping: 0=background, 1=hole, 2=knot, 3=stain  (num_classes=4)")
    A("")
    A("-" * 78)
    A(" MODEL CONFIGURATION (final model)")
    A("-" * 78)
    e = ctx["final_exp"]
    A(f"   architecture          : fasterrcnn_resnet50_fpn (COCO-pretrained backbone)")
    A(f"   num_classes           : {CONFIG['num_classes']}")
    A(f"   min_size / max_size   : {e['min_size']} / {e['max_size']}")
    A(f"   anchor aspect ratios  : {CONFIG['aspect_ratios']}")
    A(f"   trainable bb layers   : {CONFIG['trainable_backbone_layers']} (full fine-tune)")
    A(f"   optimizer             : AdamW  lr={e['lr']:g} "
      f"(backbone x{CONFIG['backbone_lr_mult']})  wd={CONFIG['weight_decay']:g}")
    A(f"   schedule              : {CONFIG['warmup_epochs']}-epoch warm-up + cosine annealing")
    A(f"   batch size            : {ctx['batch_size']} x {ctx['grad_accum']} accumulation "
      f"= effective {ctx['batch_size']*ctx['grad_accum']}")
    A(f"   AMP                   : {ctx['env'].get('amp')}")
    A(f"   gradient clipping     : {CONFIG['grad_clip_norm']}")
    A(f"   max epochs / patience : {CONFIG['epochs']} / {CONFIG['early_stop_patience']}")
    A(f"   augmentation (train)  : {CONFIG['aug']}")
    A(f"   val/test augmentation : NONE (deterministic)")
    A("")
    A("-" * 78)
    A(" EXPERIMENT COMPARISON (validation)")
    A("-" * 78)
    A(f"   {'experiment':<10}{'res':<14}{'best ep':>8}{'mAP@50':>10}{'mAP@50:95':>11}"
      f"{'weakest AP@50':>16}{'F1':>9}")
    for r in ctx["experiments"]:
        b = r["best"]
        A(f"   {r['name']:<10}{str(r['exp']['min_size'])+'/'+str(r['exp']['max_size']):<14}"
          f"{b['epoch']:>8}{100*b['map50']:>9.2f}%{100*b['map5095']:>10.2f}%"
          f"{100*b['weakest']:>15.2f}%{100*b['f1']:>8.2f}%   ({b.get('weakest_class','')})")
        A(f"   {'':<10}stopped: {r['stop_reason']}")
    A(f"   -> selected: {ctx['final_exp_name']}")
    A("")
    A("-" * 78)
    A(" CHECKPOINT SELECTION (validation only)")
    A("-" * 78)
    b = ctx["best"]
    A(f"   best epoch                 : {b['epoch']}")
    A(f"   best validation mAP@50     : {100*b['map50']:.2f}%")
    A(f"   best validation mAP@50:95  : {100*b['map5095']:.2f}%")
    A(f"   weakest class at that epoch: {b.get('weakest_class','?')} "
      f"AP@50 = {100*b['weakest']:.2f}%")
    A(f"   selection score            : {b['score']:.4f}  "
      f"(={CONFIG['sel_w_map50']}*mAP50 + {CONFIG['sel_w_weakest']}*weakestAP50 "
      f"+ {CONFIG['sel_w_map5095']}*mAP50:95)")
    A(f"   highest mAP@50 seen at all : {100*ctx['best_map50_only']['map50']:.2f}% "
      f"(epoch {ctx['best_map50_only']['epoch']})")
    A(f"   collapse guard             : a checkpoint was rejected if the weakest-class")
    A(f"                                AP@50 fell below {CONFIG['collapse_guard_ratio']:.0%} "
      f"of the best weakest-class AP@50")
    A("")
    A(f"   confidence threshold       : {ctx['conf_thresh']:.2f}  "
      f"(chosen on VALIDATION only)")
    A("")
    A("-" * 78)
    A(" VALIDATION RESULTS (best checkpoint)")
    A("-" * 78)
    A(_table_text(ctx["val_coco"], ctx["val_prf"]))
    A("")
    A("-" * 78)
    A(" TEST RESULTS  -  single evaluation, run once, after the model, epoch,")
    A("                  hyperparameters and confidence threshold were fixed")
    A("                  using the validation set only.")
    A("-" * 78)
    A(_table_text(ctx["test_coco"], ctx["test_prf"]))
    A("")
    t_o = ctx["test_prf"]["overall"]
    A(f"   Final test mAP@50      : {100*ctx['test_coco']['mAP50']:.2f}%")
    A(f"   Final test mAP@50:95   : {100*ctx['test_coco']['mAP5095']:.2f}%")
    A(f"   Final test precision   : {100*t_o['precision']:.2f}%  (conf>={ctx['conf_thresh']:.2f}, IoU>=0.50)")
    A(f"   Final test recall      : {100*t_o['recall']:.2f}%")
    A(f"   Final test F1          : {100*t_o['f1']:.2f}%")
    A(f"   Test TP/FP/FN          : {t_o['TP']} / {t_o['FP']} / {t_o['FN']}")
    if not math.isnan(ctx["test_coco"].get("AP_small", float("nan"))):
        A(f"   Test AP small/med/large: {100*ctx['test_coco']['AP_small']:.2f}% / "
          f"{100*ctx['test_coco']['AP_medium']:.2f}% / {100*ctx['test_coco']['AP_large']:.2f}%")
    A("")
    A("   Confusion matrix (test; rows = GT, cols = prediction):")
    cm = ctx["test_prf"]["confusion_matrix"]
    hdr = ["background", "hole", "knot", "stain"]
    corner = "GT \\ PRED"
    A("     " + f"{corner:<12}" + "".join(f"{h:>12}" for h in hdr))
    for i, rn in enumerate(hdr):
        A("     " + f"{rn:<12}" + "".join(f"{cm[i,j]:>12}" for j in range(4)))
    A("")
    A("-" * 78)
    A(" TERMINOLOGY")
    A("-" * 78)
    A("   AP@50 / mAP@50 are object-detection average-precision metrics, NOT accuracy.")
    A("   They are not comparable to classification Top-1 accuracy.")
    A("   Precision/recall/F1 above are computed at a fixed confidence threshold and")
    A("   IoU>=0.50 by explicit TP/FP/FN matching; AP is threshold-free (COCO).")
    A("")
    A("-" * 78)
    A(" OUTPUT FILES")
    A("-" * 78)
    for line in ctx["file_tree"]:
        A("   " + line)
    A("")
    A("=" * 78)

    with open(path, "w") as f:
        f.write("\n".join(L))
    return "\n".join(L)


def _table_text(coco_m, prf_m):
    out = []
    out.append(f"   {'Class':<10}{'AP@50':>10}{'AP@50:95':>11}{'Precision':>11}"
               f"{'Recall':>10}{'F1':>9}{'TP':>7}{'FP':>7}{'FN':>7}")
    out.append("   " + "-" * 82)
    for k in (1, 2, 3):
        c, p = coco_m["per_class"][k], prf_m["per_class"][k]
        out.append(f"   {CLASS_NAMES[k].capitalize():<10}{100*c['AP50']:>9.2f}%"
                   f"{100*c['AP5095']:>10.2f}%{100*p['precision']:>10.2f}%"
                   f"{100*p['recall']:>9.2f}%{100*p['f1']:>8.2f}%"
                   f"{p['TP']:>7}{p['FP']:>7}{p['FN']:>7}")
    out.append("   " + "-" * 82)
    o = prf_m["overall"]
    out.append(f"   {'Overall':<10}{100*coco_m['mAP50']:>9.2f}%{100*coco_m['mAP5095']:>10.2f}%"
               f"{100*o['precision']:>10.2f}%{100*o['recall']:>9.2f}%{100*o['f1']:>8.2f}%"
               f"{o['TP']:>7}{o['FP']:>7}{o['FN']:>7}")
    return "\n".join(out)


def file_tree(root, max_files_per_dir=6):
    lines = []
    for dp, dn, fn in os.walk(root):
        rel = os.path.relpath(dp, root)
        depth = 0 if rel == "." else rel.count(os.sep) + 1
        lines.append("  " * depth + (os.path.basename(dp) if rel != "." else root) + "/")
        for f in sorted(fn)[:max_files_per_dir]:
            lines.append("  " * (depth + 1) + f)
        if len(fn) > max_files_per_dir:
            lines.append("  " * (depth + 1) + f"... (+{len(fn)-max_files_per_dir} more)")
    return lines


# ==============================================================================
# 10. MAIN
# ==============================================================================
def main(zip_path=None):
    t_start = time.time()
    import torch

    out_root = CONFIG["out_dir"]
    for sub in ["", "checkpoints", "metrics", "plots", "predictions/validation",
                "predictions/test", "error_analysis/false_positives",
                "error_analysis/false_negatives", "experiments"]:
        os.makedirs(os.path.join(out_root, sub), exist_ok=True)

    device, gpu_mem_gb = setup_environment()
    seed_everything(CONFIG["seed"])

    # ---------------- data ----------------
    root, splits = discover_dataset(zip_path or CONFIG["zip_path"])
    hr("DATASET VALIDATION & CLEANING (in memory; source files untouched)")
    data = {}
    for s in ("train", "valid", "test"):
        if s in splits:
            data[s] = load_and_clean_split(s, splits[s]["json"], splits[s]["images_dir"])

    for s in data:
        assert len(data[s]["records"]) > 0, f"split {s} has no usable images"
    train_cats = set(data["train"]["class_distribution"].keys())
    log(f"  classes present in train: "
        f"{sorted([CLASS_NAMES[c] for c in train_cats])}")
    if len(train_cats) < 4:
        log("  [warn] not all four classes appear in the training annotations!")
    log("")

    # ---------------- sanity ----------------
    probe = build_model(CONFIG["num_classes"], CONFIG["experiments"][0]["min_size"],
                        CONFIG["experiments"][0]["max_size"], device)
    ok = sanity_test(probe, data["train"]["records"], device, os.path.join(out_root, "plots"))
    del probe
    if device.type == "cuda":
        torch.cuda.empty_cache()
    if not ok:
        log("SANITY TEST FAILED - aborting before training. Fix the reported issue first.")
        return None

    # ---------------- experiments ----------------
    exps = [CONFIG["experiments"][0]]
    if CONFIG["run_experiment_B"] and len(CONFIG["experiments"]) > 1:
        exps.append(CONFIG["experiments"][1])

    results = []
    for e in exps:
        if (time.time() - t_start) / 3600.0 > CONFIG["max_train_hours"]:
            log(f"[time] skipping {e['name']} - session budget exhausted. "
                f"Re-run to continue (checkpoints are on disk).")
            break
        try:
            results.append(train_one_experiment(e, data, device, gpu_mem_gb, out_root, t_start))
        except Exception as ex:
            log(f"[error] experiment {e['name']} failed: {ex}")
            log(traceback.format_exc())

    if not results:
        log("No experiment completed. Aborting.")
        return None

    all_hist = {r["name"]: r["history"] for r in results if r["history"]}
    plot_training_curves(all_hist, os.path.join(out_root, "plots"))

    # ---------------- pick the experiment (validation only) ----------------
    hr("EXPERIMENT SELECTION (validation only)")
    for r in results:
        b = r["best"]
        log(f"  {r['name']}: min_size={r['exp']['min_size']} -> "
            f"mAP@50 {100*b['map50']:.2f}%  mAP@50:95 {100*b['map5095']:.2f}%  "
            f"weakest ({b.get('weakest_class','?')}) {100*b['weakest']:.2f}%  "
            f"F1 {100*b['f1']:.2f}%  score {b['score']:.4f}  @ epoch {b['epoch']}")
    winner = max(results, key=lambda r: r["best"]["score"])
    log(f"\n  SELECTED: {winner['name']} "
        f"(highest guarded selection score on validation)")
    log("")

    # copy the winner's checkpoints to the top level
    shutil.copy2(winner["best_path"], os.path.join(out_root, "best_model.pth"))
    shutil.copy2(winner["last_path"], os.path.join(out_root, "last_model.pth"))
    src_ck = os.path.join(winner["dir"], "checkpoints")
    for f in (os.listdir(src_ck) if os.path.isdir(src_ck) else []):
        shutil.copy2(os.path.join(src_ck, f),
                     os.path.join(out_root, "checkpoints", f"{winner['name']}_{f}"))

    # ---------------- reload best model ----------------
    hr("LOADING BEST CHECKPOINT")
    model = build_model(CONFIG["num_classes"], winner["exp"]["min_size"],
                        winner["exp"]["max_size"], device)
    ck = torch.load(os.path.join(out_root, "best_model.pth"), map_location=device,
                    weights_only=False)
    model.load_state_dict(ck["model"])
    model.eval()
    log(f"  loaded epoch {ck['best']['epoch']} "
        f"(val mAP@50 {100*ck['best']['map50']:.2f}%)")
    log("")

    use_amp = CONFIG["amp"] and device.type == "cuda"

    # ---------------- validation: threshold sweep ----------------
    val_preds = run_inference(model, data["valid"]["records"], device, use_amp, "valid-final")
    conf_thresh, sweep = sweep_confidence(data["valid"], val_preds, out_root)

    val_eval = evaluate_split(model, data["valid"], device, conf_thresh, use_amp,
                              "valid-final", preds=val_preds)
    print_metric_table(val_eval["coco"], val_eval["prf"],
                       f"FINAL VALIDATION METRICS (best checkpoint, conf>={conf_thresh:.2f})")
    print_confusion(val_eval["prf"]["confusion_matrix"])
    save_per_class_csv(val_eval["coco"], val_eval["prf"], os.path.join(out_root, "metrics"),
                       "validation_best_checkpoint")
    plot_pr_curves(val_eval["coco"], os.path.join(out_root, "plots"), "validation")
    plot_confusion(val_eval["prf"]["confusion_matrix"], os.path.join(out_root, "plots"),
                   "validation")
    save_visualisations(data["valid"], val_eval["prf"],
                        os.path.join(out_root, "predictions", "validation"),
                        CONFIG["n_vis_val"], "validation")
    err_info = save_error_analysis(data["valid"], val_eval["prf"], out_root,
                                  CONFIG["n_err_examples"], "validation")

    # ---------------- TEST: exactly once ----------------
    test_coco = test_prf = None
    if "test" in data:
        hr("FINAL TEST EVALUATION  (first and only time the test set is used)")
        log("  Model, epoch, resolution, augmentation and confidence threshold were all")
        log("  fixed using the validation set before this point.")
        log("")
        test_preds = run_inference(model, data["test"]["records"], device, use_amp, "test")
        test_eval = evaluate_split(model, data["test"], device, conf_thresh, use_amp,
                                   "test", preds=test_preds)
        test_coco, test_prf = test_eval["coco"], test_eval["prf"]
        print_metric_table(test_coco, test_prf,
                           f"TEST SET RESULTS (conf>={conf_thresh:.2f}, IoU>=0.50)")
        print_confusion(test_prf["confusion_matrix"])
        save_per_class_csv(test_coco, test_prf, os.path.join(out_root, "metrics"), "test_final")
        plot_pr_curves(test_coco, os.path.join(out_root, "plots"), "test")
        plot_confusion(test_prf["confusion_matrix"], os.path.join(out_root, "plots"), "test")
        save_visualisations(data["test"], test_prf,
                            os.path.join(out_root, "predictions", "test"),
                            CONFIG["n_vis_test"], "test")
    else:
        log("[warn] no test split found - test evaluation skipped")
        test_coco = val_eval["coco"]
        test_prf = val_eval["prf"]

    # ---------------- report ----------------
    total_time = time.time() - t_start
    import torchvision
    ctx = {
        "env": {
            "PyTorch": torch.__version__,
            "Torchvision": torchvision.__version__,
            "CUDA": str(torch.version.cuda),
            "GPU": (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"),
            "GPU memory": f"{gpu_mem_gb:.2f} GB",
            "num_workers": auto_num_workers(),
            "amp": "enabled" if use_amp else "disabled",
            "seed": CONFIG["seed"],
        },
        "root": root, "splits": splits, "data": data,
        "experiments": results, "final_exp": winner["exp"], "final_exp_name": winner["name"],
        "best": ck["best"], "best_map50_only": winner["best_map50_only"],
        "batch_size": winner["batch_size"], "grad_accum": winner["grad_accum"],
        "conf_thresh": conf_thresh,
        "val_coco": val_eval["coco"], "val_prf": val_eval["prf"],
        "test_coco": test_coco, "test_prf": test_prf,
        "total_time": total_time,
        "file_tree": [],
    }
    ctx["file_tree"] = file_tree(out_root)
    report_path = os.path.join(out_root, "final_report.txt")
    text = write_final_report(report_path, ctx)

    hr("FINAL REPORT")
    print(text)
    log("")

    with open(os.path.join(out_root, "run_log.txt"), "w") as f:
        f.write("\n".join(_LOG_LINES))

    hr("SUMMARY")
    log(f"  best epoch                : {ck['best']['epoch']}  ({winner['name']})")
    log(f"  best validation mAP@50    : {100*ck['best']['map50']:.2f}%")
    log(f"  selected conf threshold   : {conf_thresh:.2f}")
    log(f"  final test mAP@50         : {100*test_coco['mAP50']:.2f}%")
    log(f"  final test mAP@50:95      : {100*test_coco['mAP5095']:.2f}%")
    log(f"  final test precision      : {100*test_prf['overall']['precision']:.2f}%")
    log(f"  final test recall         : {100*test_prf['overall']['recall']:.2f}%")
    log(f"  final test F1             : {100*test_prf['overall']['f1']:.2f}%")
    log(f"  total time                : {fmt_hms(total_time)}")
    log(f"  outputs                   : {out_root}")
    log("")
    return ctx


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--zip", default=None, help="path to the dataset zip")
    ap.add_argument("--epochs", type=int, default=None)
    ap.add_argument("--no-exp-b", action="store_true", help="skip experiment B (960/1600)")
    ap.add_argument("--debug", type=int, default=0, help="use only N train images")
    ap.add_argument("--max-hours", type=float, default=None)
    a = ap.parse_args()
    if a.epochs:
        CONFIG["epochs"] = a.epochs
    if a.no_exp_b:
        CONFIG["run_experiment_B"] = False
    if a.debug:
        CONFIG["debug_subset"] = a.debug
    if a.max_hours:
        CONFIG["max_train_hours"] = a.max_hours
    main(a.zip)


## 3. Smoke test (optional but recommended)
2 epochs, 40 training images, one resolution only — confirms the whole pipeline
(sanity test → train → validate → checkpoint → report) runs without errors,
and prints a real, measured images/second for *your* GPU so you can estimate
full-run time (`train_images / img_per_sec` → seconds per epoch).

In [ ]:
!python fabric_frcnn_pipeline.py --debug 40 --epochs 2 --no-exp-b


## 4. Full training run

- Trains Experiment A (800/1333), then Experiment B (960/1600) unless `--no-exp-b`.
- Early stopping, patience 10–15 epochs, up to 70 epochs.
- `--max-hours` stops cleanly before the Kaggle session limit; re-run this same
  cell afterward to resume from `last_model.pth`.
- Evaluates validation every epoch, sweeps the confidence threshold on
  validation only, then runs the **test set exactly once** at the end.

Adjust the flags below as needed:
- `--no-exp-b` → skip the 960/1600 experiment (~halves total time)
- `--epochs N` → override the epoch cap
- `--max-hours H` → stop after H hours (resumable)


In [ ]:
!python fabric_frcnn_pipeline.py --max-hours 8.5


## 5. Outputs

Everything is written to `/kaggle/working/faster_rcnn_results/`:

```
best_model.pth  last_model.pth
checkpoints/
metrics/     metrics.csv, per_class_metrics.csv, confidence_sweep_validation.csv
plots/       training_loss.png, validation_map.png, precision_recall.png,
             confusion_matrix_*.png, confidence_threshold_sweep.png
predictions/ validation/ (20 images)  test/ (20 images)
error_analysis/ false_positives/  false_negatives/   (per class)
final_report.txt
run_log.txt
```


In [ ]:
from IPython.display import Image, display
import os
report = "/kaggle/working/faster_rcnn_results/final_report.txt"
if os.path.exists(report):
    print(open(report).read())
else:
    print("Run the training cell first.")


In [ ]:
# Quick visual check of validation curves
import os
p = "/kaggle/working/faster_rcnn_results/plots/validation_map.png"
if os.path.exists(p):
    from IPython.display import Image, display
    display(Image(p))
